In [ ]:
# Moirai
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import uni2ts
import torch

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import (
    MoiraiForecast,
    MoiraiModule
)

DATA_DIR = "/home/parcot1/updated_data"

TRAIN_FILE = os.path.join(DATA_DIR, "train.csv")
VAL_FILE = os.path.join(DATA_DIR, "validation.csv")
TEST_FILE = os.path.join(DATA_DIR, "test.csv")

TARGET_COL = "retreat_change_next_month"
GLACIER_COL = "glacier"
GLACIER_CODE_COL = "glacier_code"
TIME_COL = "datetime"

MODEL_NAME = "Salesforce/moirai-1.1-R-small"

CONTEXT_LENGTHS = [6, 12, 24, 36]
PREDICTION_LENGTH = 1
NUM_SAMPLES = 100
BATCH_SIZE = 1
PATCH_SIZE = "auto"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

BASE_PREDICTOR_COLUMNS = [
    "time_idx",
    "year",
    "month",
    "month_sin",
    "month_cos",
    "retreat",
    "retreat_change",
    "retreat_lag_1",
    "retreat_change_lag_1",
    "terminus_thermal",
    "shelf_thermal",
    "undercutting",
    "discharge",
    "x_epsg3413",
    "y_epsg3413",
    "basin_CE",
    "basin_CW",
    "basin_N",
    "basin_NE",
    "basin_NW",
    "basin_SE",
    "basin_SW",
    "category_CR",
    "category_DW",
    "category_FE",
    "category_NC",
    "category_SC",
    "category_SR"
]

PREDICTOR_COLUMNS = [GLACIER_CODE_COL] + [
    col for col in BASE_PREDICTOR_COLUMNS if col != TARGET_COL
]

def load_data():
    print("\nLoading data...")
    train_df = pd.read_csv(TRAIN_FILE)
    val_df = pd.read_csv(VAL_FILE)
    test_df = pd.read_csv(TEST_FILE)

    train_df.columns = train_df.columns.str.strip()
    val_df.columns = val_df.columns.str.strip()
    test_df.columns = test_df.columns.str.strip()

    for df in [train_df, val_df, test_df]:
        df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
        df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
        df.sort_values([GLACIER_COL, TIME_COL], inplace=True)
        df.reset_index(drop=True, inplace=True)

    return train_df, val_df, test_df

def create_glacier_mapping(train_df, val_df, test_df):
    all_glaciers = sorted(
        set(train_df[GLACIER_COL].dropna().unique())
        | set(val_df[GLACIER_COL].dropna().unique())
        | set(test_df[GLACIER_COL].dropna().unique())
    )

    glacier_mapping = {
        glacier: float(index)
        for index, glacier in enumerate(all_glaciers)
    }

    for df in [train_df, val_df, test_df]:
        df[GLACIER_CODE_COL] = df[GLACIER_COL].map(glacier_mapping)

    print("\nNumber of glaciers:", len(all_glaciers))
    print("\nFirst five glacier mappings:")

    mapping_preview = pd.DataFrame(
        {
            GLACIER_COL: all_glaciers[:5],
            GLACIER_CODE_COL: [glacier_mapping[g] for g in all_glaciers[:5]]
        }
    )

    print(mapping_preview.to_string(index=False))

    return train_df, val_df, test_df, all_glaciers, glacier_mapping

def prepare_predictors(train_df, val_df, test_df):
    required_columns = [TARGET_COL, GLACIER_COL, TIME_COL] + PREDICTOR_COLUMNS

    for col in required_columns:
        if col not in train_df.columns:
            raise ValueError(f"Missing column in train.csv: {col}")
        if col not in val_df.columns:
            raise ValueError(f"Missing column in validation.csv: {col}")
        if col not in test_df.columns:
            raise ValueError(f"Missing column in test.csv: {col}")

    for df in [train_df, val_df, test_df]:
        for col in PREDICTOR_COLUMNS:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")

    print("\nNumber of predictors:", len(PREDICTOR_COLUMNS))
    print("\nPredictors:")
    for col in PREDICTOR_COLUMNS:
        print(col)

    return train_df, val_df, test_df

def load_model(context_length):
    print("\nLoading:", MODEL_NAME)

    module = MoiraiModule.from_pretrained(MODEL_NAME)

    model = MoiraiForecast(
        module=module,
        prediction_length=PREDICTION_LENGTH,
        context_length=context_length,
        patch_size=PATCH_SIZE,
        num_samples=NUM_SAMPLES,
        target_dim=1,
        feat_dynamic_real_dim=len(PREDICTOR_COLUMNS),
        past_feat_dynamic_real_dim=0
    )

    predictor = model.create_predictor(
        batch_size=BATCH_SIZE,
        device=DEVICE
    )

    return predictor

def create_moirai_dataset(history, context_length):
    history = history.sort_values(TIME_COL).tail(context_length).copy()

    if len(history) < context_length:
        return None

    required_columns = [TIME_COL, TARGET_COL] + PREDICTOR_COLUMNS
    history = history.dropna(subset=required_columns).copy()

    if len(history) < context_length:
        return None

    for col in PREDICTOR_COLUMNS:
        history[col] = pd.to_numeric(history[col], errors="coerce")

    history[TARGET_COL] = pd.to_numeric(history[TARGET_COL], errors="coerce")

    if history[required_columns].isna().any().any():
        return None

    target_values = history[TARGET_COL].to_numpy(dtype=np.float32)
    feature_values = history[PREDICTOR_COLUMNS].to_numpy(dtype=np.float32).T

    start_date = history[TIME_COL].iloc[0]
    glacier_name = str(history[GLACIER_COL].iloc[0])

    item = {
        "item_id": glacier_name,
        "start": pd.Period(start_date, freq="M"),
        "target": target_values,
        "feat_dynamic_real": feature_values
    }

    dataset = ListDataset([item], freq="M")
    return dataset

def predict_one_step(predictor, history, context_length):
    dataset = create_moirai_dataset(history, context_length)

    if dataset is None:
        return None

    try:
        forecast_iterator = predictor.predict(dataset)
        forecast = next(iter(forecast_iterator))
        samples = np.asarray(forecast.samples)
        prediction = float(np.median(samples))
        return prediction
    except Exception as e:
        glacier_name = history[GLACIER_COL].iloc[0] if len(history) > 0 else "unknown"
        print("\nPrediction failed for", glacier_name, ":", repr(e))
        return None

def calculate_metrics(actuals, predictions):
    if len(actuals) == 0:
        return {
            "RMSE": np.nan,
            "MSE": np.nan,
            "MAE": np.nan,
            "R2": np.nan,
            "Predictions": 0
        }

    actuals = np.asarray(actuals, dtype=float)
    predictions = np.asarray(predictions, dtype=float)

    mse = mean_squared_error(actuals, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actuals, predictions)

    if len(np.unique(actuals)) > 1:
        r2 = r2_score(actuals, predictions)
    else:
        r2 = np.nan

    return {
        "RMSE": rmse,
        "MSE": mse,
        "MAE": mae,
        "R2": r2,
        "Predictions": len(predictions)
    }

def evaluate_split(history_df, evaluation_df, predictor, context_length, split_name):
    print("\nEvaluating:", split_name)

    predictions = []
    actuals = []

    common_glaciers = sorted(
        set(history_df[GLACIER_COL].dropna().unique())
        & set(evaluation_df[GLACIER_COL].dropna().unique())
    )

    for glacier in common_glaciers:
        glacier_history = history_df[
            history_df[GLACIER_COL] == glacier
        ].sort_values(TIME_COL).copy()

        glacier_evaluation = evaluation_df[
            evaluation_df[GLACIER_COL] == glacier
        ].sort_values(TIME_COL).copy()

        if len(glacier_history) < context_length:
            continue

        for _, row in glacier_evaluation.iterrows():
            actual = row[TARGET_COL]

            if pd.isna(actual):
                continue

            prediction = predict_one_step(
                predictor=predictor,
                history=glacier_history,
                context_length=context_length
            )

            if prediction is None:
                continue

            predictions.append(prediction)
            actuals.append(float(actual))

            glacier_history = pd.concat(
                [glacier_history, row.to_frame().T],
                ignore_index=True
            )

    metrics = calculate_metrics(actuals, predictions)

    print(split_name, "RMSE:", metrics["RMSE"])
    print(split_name, "MSE:", metrics["MSE"])
    print(split_name, "MAE:", metrics["MAE"])
    print(split_name, "R2:", metrics["R2"])
    print(split_name, "Predictions:", metrics["Predictions"])

    return metrics

def main():
    train_df, val_df, test_df = load_data()

    train_df, val_df, test_df, all_glaciers, glacier_mapping = create_glacier_mapping(
        train_df, val_df, test_df
    )

    train_df, val_df, test_df = prepare_predictors(
        train_df, val_df, test_df
    )

    results = []

    for context_length in CONTEXT_LENGTHS:
        print("\n" + "=" * 60)
        print("Context Length:", context_length)
        print("=" * 60)

        predictor = load_model(context_length)

        validation_metrics = evaluate_split(
            history_df=train_df,
            evaluation_df=val_df,
            predictor=predictor,
            context_length=context_length,
            split_name="Validation"
        )

        train_val_df = pd.concat([train_df, val_df], ignore_index=True)
        train_val_df = train_val_df.sort_values(
            [GLACIER_COL, TIME_COL]
        ).reset_index(drop=True)

        test_metrics = evaluate_split(
            history_df=train_val_df,
            evaluation_df=test_df,
            predictor=predictor,
            context_length=context_length,
            split_name="Test"
        )

        results.append(
            {
                "Context Length": context_length,
                "Validation RMSE": validation_metrics["RMSE"],
                "Validation MSE": validation_metrics["MSE"],
                "Validation MAE": validation_metrics["MAE"],
                "Validation R2": validation_metrics["R2"],
                "Validation Predictions": validation_metrics["Predictions"],
                "Test RMSE": test_metrics["RMSE"],
                "Test MSE": test_metrics["MSE"],
                "Test MAE": test_metrics["MAE"],
                "Test R2": test_metrics["R2"],
                "Test Predictions": test_metrics["Predictions"]
            }
        )

    results_df = pd.DataFrame(results)

    print("\n\nFinal Results")
    print(results_df.to_string(index=False))
    results_df = pd.DataFrame(results)

    print("\n\nFinal Results")
    print(results_df.to_string(index=False))

    output_file = os.path.join(DATA_DIR, "moirai_results.csv")
    results_df.to_csv(output_file, index=False)
    print("\nSaved results to:", output_file)

if __name__ == "__main__":
    main()

In [ ]:
# Chronos2

import os
import warnings
import random

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from chronos import Chronos2Pipeline

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = "/home/parcot1/updated_data"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
VALIDATION_PATH = os.path.join(DATA_DIR, "validation.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
OUTPUT_DIR = os.path.join(DATA_DIR, "Chronos2_results")

os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COL = "retreat_change_next_month"
GLACIER_COL = "glacier"
TIME_COL = "datetime"
ITEM_ID_COL = "item_id"
PREDICTION_LENGTH = 1
CONTEXT_LENGTHS = [6, 12, 24, 36]

COVARIATE_COLS = [
    "time_idx",
    "year",
    "month",
    "month_sin",
    "month_cos",
    "retreat",
    "retreat_change",
    "retreat_lag_1",
    "retreat_change_lag_1",
    "terminus_thermal",
    "shelf_thermal",
    "undercutting",
    "discharge",
    "x_epsg3413",
    "y_epsg3413",
    "basin_CE",
    "basin_CW",
    "basin_N",
    "basin_NE",
    "basin_NW",
    "basin_SE",
    "basin_SW",
    "category_CR",
    "category_DW",
    "category_FE",
    "category_NC",
    "category_SC",
    "category_SR"
]

def load_data(path):
    df = pd.read_csv(path)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df[GLACIER_COL] = df[GLACIER_COL].astype(str).str.strip()
    return df

train_df = load_data(TRAIN_PATH)
val_df = load_data(VALIDATION_PATH)
test_df = load_data(TEST_PATH)

def clean_data(df):
    df = df.copy()
    df = (
        df.sort_values([GLACIER_COL, TIME_COL])
          .drop_duplicates(subset=[GLACIER_COL, TIME_COL], keep="last")
          .reset_index(drop=True)
    )
    return df

train_df = clean_data(train_df)
val_df = clean_data(val_df)
test_df = clean_data(test_df)

all_glaciers = pd.concat(
    [train_df[[GLACIER_COL]], val_df[[GLACIER_COL]], test_df[[GLACIER_COL]]],
    ignore_index=True
)

unique_glaciers = sorted(all_glaciers[GLACIER_COL].dropna().unique())

glacier_mapping = {
    glacier: str(code)
    for code, glacier in enumerate(unique_glaciers)
}

def add_item_id(df):
    df = df.copy()
    df[ITEM_ID_COL] = df[GLACIER_COL].map(glacier_mapping)
    return df

train_df = add_item_id(train_df)
val_df = add_item_id(val_df)
test_df = add_item_id(test_df)

required_columns = [TARGET_COL, GLACIER_COL, TIME_COL, ITEM_ID_COL] + COVARIATE_COLS

def validate_columns(df, name):
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"{name} is missing columns: {missing_columns}")

validate_columns(train_df, "Training data")
validate_columns(val_df, "Validation data")
validate_columns(test_df, "Test data")

numeric_columns = [TARGET_COL] + COVARIATE_COLS

def convert_numeric(df):
    df = df.copy()
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

train_df = convert_numeric(train_df)
val_df = convert_numeric(val_df)
test_df = convert_numeric(test_df)

training_fill_values = {col: train_df[col].median() for col in COVARIATE_COLS}

def apply_causal_missing_value_policy(df):
    df = df.copy()
    df = df.sort_values([GLACIER_COL, TIME_COL]).reset_index(drop=True)
    for col in COVARIATE_COLS:
        df[col] = df.groupby(GLACIER_COL)[col].transform(lambda x: x.ffill())
        df[col] = df[col].fillna(training_fill_values[col])
    df = df.dropna(subset=[TARGET_COL])
    return df

train_df = apply_causal_missing_value_policy(train_df)
val_df = apply_causal_missing_value_policy(val_df)
test_df = apply_causal_missing_value_policy(test_df)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("Number of glaciers:", len(glacier_mapping))

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=device
)

def calculate_metrics(results_df):
    if results_df is None or len(results_df) == 0:
        return {"RMSE": np.nan, "MSE": np.nan, "MAE": np.nan, "R2": np.nan, "N": 0}

    y_true = results_df["actual"].astype(float).values
    y_pred = results_df["prediction"].astype(float).values

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)

    if len(np.unique(y_true)) > 1:
        r2 = r2_score(y_true, y_pred)
    else:
        r2 = np.nan

    return {
        "RMSE": float(rmse),
        "MSE": float(mse),
        "MAE": float(mae),
        "R2": float(r2),
        "N": int(len(results_df))
    }

def prepare_chronos_dataframe(df):
    output_columns = [ITEM_ID_COL, TIME_COL, TARGET_COL] + COVARIATE_COLS
    output = (
        df[output_columns]
        .copy()
        .sort_values([ITEM_ID_COL, TIME_COL])
        .reset_index(drop=True)
    )
    output[ITEM_ID_COL] = output[ITEM_ID_COL].astype(str)
    return output

def forecast_chronos2(pipeline, history_df, forecast_df, context_length, split_name):
    history = history_df.copy()
    forecast = forecast_df.copy()

    history = history.sort_values([GLACIER_COL, TIME_COL]).reset_index(drop=True)
    forecast = forecast.sort_values([GLACIER_COL, TIME_COL]).reset_index(drop=True)

    results = []
    glaciers = sorted(forecast[GLACIER_COL].dropna().unique())

    print(f"FORECASTING {split_name}")
    print("Context length:", context_length)
    print("Number of glaciers:", len(glaciers))
    print("Rows to forecast:", len(forecast))

    processed = 0

    for glacier in glaciers:
        glacier_history = history[history[GLACIER_COL] == glacier].sort_values(TIME_COL).copy()
        glacier_forecast = forecast[forecast[GLACIER_COL] == glacier].sort_values(TIME_COL).copy()

        if len(glacier_history) < context_length:
            continue

        for _, forecast_row in glacier_forecast.iterrows():
            context = glacier_history.tail(context_length).copy()
            context_input = prepare_chronos_dataframe(context)

            try:
                prediction_df = pipeline.predict_df(
                    context_input,
                    prediction_length=1,
                    quantile_levels=[0.1, 0.5, 0.9],
                    id_column=ITEM_ID_COL,
                    timestamp_column=TIME_COL,
                    target=TARGET_COL
                )
                prediction = float(prediction_df["0.5"].iloc[0])
            except Exception:
                continue

            results.append({
                "glacier": glacier,
                "item_id": str(forecast_row[ITEM_ID_COL]),
                "datetime": forecast_row[TIME_COL],
                "actual": float(forecast_row[TARGET_COL]),
                "prediction": prediction,
                "split": split_name,
                "context_length": context_length
            })

            glacier_history = pd.concat([glacier_history, pd.DataFrame([forecast_row])], ignore_index=True)
            glacier_history = (
                glacier_history
                .drop_duplicates(subset=[GLACIER_COL, TIME_COL], keep="last")
                .sort_values(TIME_COL)
                .reset_index(drop=True)
            )

            processed += 1

    return pd.DataFrame(results)

validation_results = []

for context_length in CONTEXT_LENGTHS:
    validation_predictions = forecast_chronos2(
        pipeline=pipeline,
        history_df=train_df,
        forecast_df=val_df,
        context_length=context_length,
        split_name="Validation"
    )

    validation_prediction_path = os.path.join(
        OUTPUT_DIR,
        f"Chronos2_validation_predictions_context_{context_length}.csv"
    )
    validation_predictions.to_csv(validation_prediction_path, index=False)

    metrics = calculate_metrics(validation_predictions)

    validation_results.append({
        "Model": "Chronos2",
        "Split": "Validation",
        "ContextLength": context_length,
        **metrics
    })

validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values("ContextLength")
    .reset_index(drop=True)
)

validation_results_df.to_csv(
    os.path.join(OUTPUT_DIR, "Chronos2_validation_results.csv"),
    index=False
)

valid_context_results = validation_results_df.dropna(subset=["RMSE"])

if len(valid_context_results) == 0:
    raise RuntimeError("No valid validation results were generated.")

best_context = int(
    valid_context_results.sort_values("RMSE", ascending=True).iloc[0]["ContextLength"]
)

best_validation_rmse = float(
    valid_context_results.sort_values("RMSE", ascending=True).iloc[0]["RMSE"]
)

train_val_df = pd.concat([train_df, val_df], ignore_index=True)
train_val_df = (
    train_val_df
    .drop_duplicates(subset=[GLACIER_COL, TIME_COL], keep="last")
    .sort_values([GLACIER_COL, TIME_COL])
    .reset_index(drop=True)
)

test_predictions = forecast_chronos2(
    pipeline=pipeline,
    history_df=train_val_df,
    forecast_df=test_df,
    context_length=best_context,
    split_name="Test"
)

test_prediction_path = os.path.join(
    OUTPUT_DIR,
    f"Chronos2_test_predictions_context_{best_context}.csv"
)
test_predictions.to_csv(test_prediction_path, index=False)

test_metrics = calculate_metrics(test_predictions)

test_results_df = pd.DataFrame([
    {
        "Model": "Chronos2",
        "Split": "Test",
        "ContextLength": best_context,
        **test_metrics
    }
])

test_results_df.to_csv(
    os.path.join(OUTPUT_DIR, "Chronos2_final_test_results.csv"),
    index=False
)

experiment_summary = pd.DataFrame([
    {
        "Model": "Chronos2",
        "Target": TARGET_COL,
        "Train_Period": "1992-2011",
        "Validation_Period": "2012-2014",
        "Test_Period": "2015-2017",
        "Context_Lengths_Tested": "6, 12, 24, 36",
        "Selected_Context": best_context,
        "Validation_RMSE": best_validation_rmse,
        "Test_RMSE": test_metrics["RMSE"],
        "Test_MSE": test_metrics["MSE"],
        "Test_MAE": test_metrics["MAE"],
        "Test_R2": test_metrics["R2"],
        "Test_N": test_metrics["N"]
    }
])

experiment_summary.to_csv(
    os.path.join(OUTPUT_DIR, "Chronos2_experiment_summary.csv"),
    index=False
)

print(validation_results_df.to_string(index=False))
print(test_results_df.to_string(index=False))
print(experiment_summary.to_string(index=False))

In [ ]:
# linear reg
import os
import warnings
import random

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


SEED = 42

random.seed(SEED)
np.random.seed(SEED)


DATA_DIR = "/home/parcot1/updated_data"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
VALIDATION_PATH = os.path.join(DATA_DIR, "validation.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

OUTPUT_DIR = os.path.join(
    DATA_DIR,
    "LinearRegression_results"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


TARGET_COL = "retreat_change_next_month"
GLACIER_COL = "glacier"
GLACIER_CODE_COL = "glacier_code"
TIME_COL = "datetime"

CONTEXT_LENGTHS = [6, 12, 24, 36]


PREDICTOR_COLS = [
    "glacier_code",
    "time_idx",
    "year",
    "month",
    "month_sin",
    "month_cos",
    "retreat",
    "retreat_change",
    "retreat_lag_1",
    "retreat_change_lag_1",
    "terminus_thermal",
    "shelf_thermal",
    "undercutting",
    "discharge",
    "x_epsg3413",
    "y_epsg3413",
    "basin_CE",
    "basin_CW",
    "basin_N",
    "basin_NE",
    "basin_NW",
    "basin_SE",
    "basin_SW",
    "category_CR",
    "category_DW",
    "category_FE",
    "category_NC",
    "category_SC",
    "category_SR"
]


def load_data(path):
    df = pd.read_csv(path)

    df[TIME_COL] = pd.to_datetime(
        df[TIME_COL],
        errors="coerce"
    )

    df[GLACIER_COL] = (
        df[GLACIER_COL]
        .astype(str)
        .str.strip()
    )

    return df


train_df = load_data(TRAIN_PATH)
val_df = load_data(VALIDATION_PATH)
test_df = load_data(TEST_PATH)


all_glaciers = pd.concat(
    [
        train_df[[GLACIER_COL]],
        val_df[[GLACIER_COL]],
        test_df[[GLACIER_COL]]
    ],
    ignore_index=True
)


unique_glaciers = sorted(
    all_glaciers[GLACIER_COL]
    .dropna()
    .unique()
)


glacier_mapping = {
    glacier: code
    for code, glacier in enumerate(unique_glaciers)
}


print(
    "Number of glaciers:",
    len(glacier_mapping)
)


def add_glacier_code(df):
    df = df.copy()

    df[GLACIER_CODE_COL] = (
        df[GLACIER_COL]
        .map(glacier_mapping)
    )

    return df


train_df = add_glacier_code(train_df)
val_df = add_glacier_code(val_df)
test_df = add_glacier_code(test_df)


required_columns = (
    [
        TARGET_COL,
        GLACIER_COL,
        GLACIER_CODE_COL,
        TIME_COL
    ]
    + PREDICTOR_COLS
)


def validate_columns(df, name):
    missing = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{name} is missing columns: {missing}"
        )


validate_columns(
    train_df,
    "Training data"
)

validate_columns(
    val_df,
    "Validation data"
)

validate_columns(
    test_df,
    "Test data"
)


def clean_data(df):
    df = df.copy()

    df = df.dropna(
        subset=[
            TIME_COL,
            GLACIER_COL,
            GLACIER_CODE_COL,
            TARGET_COL
        ]
    )

    df = (
        df
        .sort_values(
            [
                GLACIER_COL,
                TIME_COL
            ]
        )
        .reset_index(drop=True)
    )

    return df


train_df = clean_data(train_df)
val_df = clean_data(val_df)
test_df = clean_data(test_df)


numeric_columns = list(
    set(
        PREDICTOR_COLS
        + [TARGET_COL]
    )
)


def convert_numeric(df):
    df = df.copy()

    for col in numeric_columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    return df


train_df = convert_numeric(train_df)
val_df = convert_numeric(val_df)
test_df = convert_numeric(test_df)


def apply_missing_value_policy(df):
    df = df.copy()

    df = (
        df
        .sort_values(
            [
                GLACIER_COL,
                TIME_COL
            ]
        )
        .reset_index(drop=True)
    )

    for col in PREDICTOR_COLS:

        if col == GLACIER_CODE_COL:
            continue

        df[col] = (
            df
            .groupby(GLACIER_COL)[col]
            .transform(
                lambda x: x.ffill().bfill()
            )
        )

        df[col] = df[col].fillna(0.0)

    df = df.dropna(
        subset=[TARGET_COL]
    )

    return df


train_df = apply_missing_value_policy(train_df)
val_df = apply_missing_value_policy(val_df)
test_df = apply_missing_value_policy(test_df)


def create_window_features(
    history_df,
    context_length
):

    rows = []

    history_df = (
        history_df
        .sort_values(
            [
                GLACIER_COL,
                TIME_COL
            ]
        )
        .reset_index(drop=True)
    )

    for glacier, glacier_df in history_df.groupby(
        GLACIER_COL,
        sort=False
    ):

        glacier_df = (
            glacier_df
            .sort_values(TIME_COL)
            .reset_index(drop=True)
        )

        for i in range(
            context_length,
            len(glacier_df)
        ):

            current_row = glacier_df.iloc[i]

            history_window = glacier_df.iloc[
                i - context_length:i
            ]

            feature_row = {}

            for col in PREDICTOR_COLS:

                if col == GLACIER_CODE_COL:

                    feature_row[
                        f"{col}_current"
                    ] = float(
                        current_row[col]
                    )

                else:

                    values = (
                        history_window[col]
                        .astype(float)
                        .values
                    )

                    for j, value in enumerate(
                        values,
                        start=1
                    ):

                        feature_row[
                            f"{col}_lag_"
                            f"{context_length - j + 1}"
                        ] = float(value)

            feature_row[TARGET_COL] = float(
                current_row[TARGET_COL]
            )

            feature_row[GLACIER_COL] = glacier

            feature_row[TIME_COL] = current_row[
                TIME_COL
            ]

            rows.append(feature_row)

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)


def train_linear_regression(
    train_features
):

    X_train = train_features.drop(
        columns=[
            TARGET_COL,
            GLACIER_COL,
            TIME_COL
        ]
    )

    y_train = train_features[
        TARGET_COL
    ]

    model = LinearRegression()

    model.fit(
        X_train,
        y_train
    )

    return model


def rolling_forecast(
    model,
    initial_history,
    forecast_df,
    context_length,
    split_name
):

    history = (
        initial_history
        .copy()
        .sort_values(
            [
                GLACIER_COL,
                TIME_COL
            ]
        )
        .reset_index(drop=True)
    )

    forecast_df = (
        forecast_df
        .copy()
        .sort_values(
            [
                TIME_COL,
                GLACIER_COL
            ]
        )
        .reset_index(drop=True)
    )

    results = []

    for _, row in forecast_df.iterrows():

        glacier = row[GLACIER_COL]

        glacier_history = (
            history[
                history[GLACIER_COL] == glacier
            ]
            .sort_values(TIME_COL)
        )

        if len(glacier_history) < context_length:
            continue

        context = glacier_history.tail(
            context_length
        )

        feature_row = {}

        for col in PREDICTOR_COLS:

            if col == GLACIER_CODE_COL:

                feature_row[
                    f"{col}_current"
                ] = float(row[col])

            else:

                values = (
                    context[col]
                    .astype(float)
                    .values
                )

                for j, value in enumerate(
                    values,
                    start=1
                ):

                    feature_row[
                        f"{col}_lag_"
                        f"{context_length - j + 1}"
                    ] = float(value)

        X = pd.DataFrame(
            [feature_row]
        )

        prediction = float(
            model.predict(X)[0]
        )

        results.append(
            {
                "glacier": glacier,
                "glacier_code": int(
                    row[GLACIER_CODE_COL]
                ),
                "datetime": row[TIME_COL],
                "actual": float(
                    row[TARGET_COL]
                ),
                "prediction": prediction,
                "split": split_name,
                "context_length": context_length
            }
        )

        history = pd.concat(
            [
                history,
                pd.DataFrame([row])
            ],
            ignore_index=True
        )

        history = (
            history
            .drop_duplicates(
                subset=[
                    GLACIER_COL,
                    TIME_COL
                ],
                keep="last"
            )
            .sort_values(
                [
                    GLACIER_COL,
                    TIME_COL
                ]
            )
            .reset_index(drop=True)
        )

    return pd.DataFrame(results)


def calculate_metrics(
    results_df
):

    if (
        results_df is None
        or len(results_df) == 0
    ):

        return {
            "RMSE": np.nan,
            "MSE": np.nan,
            "MAE": np.nan,
            "R2": np.nan,
            "N": 0
        }

    y_true = (
        results_df["actual"]
        .astype(float)
        .values
    )

    y_pred = (
        results_df["prediction"]
        .astype(float)
        .values
    )

    mse = mean_squared_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = (
        r2_score(
            y_true,
            y_pred
        )
        if len(np.unique(y_true)) > 1
        else np.nan
    )

    return {
        "RMSE": float(rmse),
        "MSE": float(mse),
        "MAE": float(mae),
        "R2": float(r2),
        "N": int(len(results_df))
    }


validation_results = []


for context_length in CONTEXT_LENGTHS:

    print(
        f"\nLinear Regression Validation "
        f"Context = {context_length}"
    )

    train_features = create_window_features(
        train_df,
        context_length
    )

    model = train_linear_regression(
        train_features
    )

    val_predictions = rolling_forecast(
        model,
        train_df,
        val_df,
        context_length,
        "Validation"
    )

    val_predictions.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"LinearRegression_validation_predictions_"
            f"context_{context_length}.csv"
        ),
        index=False
    )

    metrics = calculate_metrics(
        val_predictions
    )

    validation_results.append(
        {
            "Model": "LinearRegression",
            "Split": "Validation",
            "ContextLength": context_length,
            **metrics
        }
    )

    print(
        f"Validation RMSE: {metrics['RMSE']:.6f}"
    )

    print(
        f"Validation MSE: {metrics['MSE']:.6f}"
    )

    print(
        f"Validation MAE: {metrics['MAE']:.6f}"
    )

    print(
        f"Validation R2: {metrics['R2']:.6f}"
    )


validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values("ContextLength")
    .reset_index(drop=True)
)


print(
    "\nLinear Regression Validation Results"
)

print(
    validation_results_df.to_string(
        index=False
    )
)


validation_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "LinearRegression_validation_results.csv"
    ),
    index=False
)


best_context = int(
    validation_results_df
    .sort_values("RMSE")
    .iloc[0]["ContextLength"]
)


print(
    "\nBest Linear Regression context:",
    best_context
)


train_val_df = pd.concat(
    [
        train_df,
        val_df
    ],
    ignore_index=True
)


train_val_df = (
    train_val_df
    .drop_duplicates(
        subset=[
            GLACIER_COL,
            TIME_COL
        ],
        keep="last"
    )
    .sort_values(
        [
            GLACIER_COL,
            TIME_COL
        ]
    )
    .reset_index(drop=True)
)


test_results = []


for context_length in CONTEXT_LENGTHS:

    print(
        f"\nLinear Regression Test "
        f"Context = {context_length}"
    )

    train_val_features = create_window_features(
        train_val_df,
        context_length
    )

    model = train_linear_regression(
        train_val_features
    )

    test_predictions = rolling_forecast(
        model,
        train_val_df,
        test_df,
        context_length,
        "Test"
    )

    test_predictions.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"LinearRegression_test_predictions_"
            f"context_{context_length}.csv"
        ),
        index=False
    )

    metrics = calculate_metrics(
        test_predictions
    )

    test_results.append(
        {
            "Model": "LinearRegression",
            "Split": "Test",
            "ContextLength": context_length,
            **metrics,
            "SelectedByValidation": (
                context_length == best_context
            )
        }
    )

    print(
        f"Test RMSE: {metrics['RMSE']:.6f}"
    )

    print(
        f"Test MSE: {metrics['MSE']:.6f}"
    )

    print(
        f"Test MAE: {metrics['MAE']:.6f}"
    )

    print(
        f"Test R2: {metrics['R2']:.6f}"
    )


test_results_df = (
    pd.DataFrame(test_results)
    .sort_values("ContextLength")
    .reset_index(drop=True)
)


print(
    "\nLinear Regression Test Results"
)

print(
    test_results_df.to_string(
        index=False
    )
)


test_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "LinearRegression_test_results_all_contexts.csv"
    ),
    index=False
)


best_test_result = (
    test_results_df[
        test_results_df["ContextLength"] == best_context
    ]
)


best_test_result.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "LinearRegression_best_context_test_result.csv"
    ),
    index=False
)


experiment_summary = pd.DataFrame(
    [
        {
            "Model": "LinearRegression",
            "Train_Period": "1992-2011",
            "Validation_Period": "2012-2014",
            "Test_Period": "2015-2017",
            "Selected_Context": best_context,
            "Validation_RMSE": float(
                validation_results_df.loc[
                    validation_results_df["ContextLength"]
                    == best_context,
                    "RMSE"
                ].iloc[0]
            ),
            "Test_RMSE": float(
                best_test_result["RMSE"].iloc[0]
            ),
            "Test_MAE": float(
                best_test_result["MAE"].iloc[0]
            ),
            "Test_R2": float(
                best_test_result["R2"].iloc[0]
            ),
            "Test_N": int(
                best_test_result["N"].iloc[0]
            )
        }
    ]
)


experiment_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "LinearRegression_experiment_summary.csv"
    ),
    index=False
)


print(
    "\nFinal Linear Regression Experiment Summary"
)

print(
    experiment_summary.to_string(
        index=False
    )
)

print(
    "\nResults saved to:",
    OUTPUT_DIR
)

In [ ]:
# lightgbm
import os
import warnings
import random

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Paths
data_dir = "/home/parcot1/updated_data"
train_path = os.path.join(data_dir, "train.csv")
val_path = os.path.join(data_dir, "validation.csv")
test_path = os.path.join(data_dir, "test.csv")


# Settings
target_col = "retreat_change_next_month"
raw_id_col = "glacier"
series_id_col = "glacier_code"
time_col = "datetime"

CONTEXT_LENGTHS = [6, 12, 24, 36]
PREDICTION_LENGTH = 1
SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# Required predictors

base_feature_cols = [
    series_id_col,
    "time_idx",
    "year",
    "month",
    "month_sin",
    "month_cos",
    "retreat",
    "retreat_change",
    "retreat_lag_1",
    "retreat_change_lag_1",
    "terminus_thermal",
    "shelf_thermal",
    "undercutting",
    "discharge",
    "x_epsg3413",
    "y_epsg3413",
    "basin_CE",
    "basin_CW",
    "basin_N",
    "basin_NE",
    "basin_NW",
    "basin_SE",
    "basin_SW",
    "category_CR",
    "category_DW",
    "category_FE",
    "category_NC",
    "category_SC",
    "category_SR",
]


# Load data

print("Loading datasets")
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)


# Basic preparation
def basic_prep(df):
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df[raw_id_col] = df[raw_id_col].astype(str).str.strip()
    df = df.dropna(subset=[time_col, raw_id_col, target_col]).copy()
    df = df.sort_values([raw_id_col, time_col]).reset_index(drop=True)
    return df


train_df = basic_prep(train_df)
val_df = basic_prep(val_df)
test_df = basic_prep(test_df)


# Keep common glaciers only

train_ids = set(train_df[raw_id_col].unique())
val_ids = set(val_df[raw_id_col].unique())
test_ids = set(test_df[raw_id_col].unique())
common_ids = train_ids & val_ids & test_ids

print("\nTrain glaciers:", len(train_ids))
print("Validation glaciers:", len(val_ids))
print("Test glaciers:", len(test_ids))
print("Common glaciers:", len(common_ids))

train_panel = train_df[train_df[raw_id_col].isin(common_ids)].copy()
val_panel = val_df[val_df[raw_id_col].isin(common_ids)].copy()
test_panel = test_df[test_df[raw_id_col].isin(common_ids)].copy()


# Glacier encoding

all_glaciers = sorted(
    pd.concat(
        [
            train_panel[[raw_id_col]],
            val_panel[[raw_id_col]],
            test_panel[[raw_id_col]],
        ],
        ignore_index=True
    )[raw_id_col].dropna().astype(str).unique()
)

glacier_code_map = {g: i for i, g in enumerate(all_glaciers)}

for df in [train_panel, val_panel, test_panel]:
    df[series_id_col] = df[raw_id_col].map(glacier_code_map).astype(int)


# Numeric cleanup
for df in [train_panel, val_panel, test_panel]:
    for col in base_feature_cols + [target_col]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

# Final feature set

feature_cols = [
    col for col in base_feature_cols
    if col in train_panel.columns
    and col in val_panel.columns
    and col in test_panel.columns
]

missing_from_any_split = [col for col in base_feature_cols if col not in feature_cols]
if missing_from_any_split:
    print("\nDropped missing feature columns:", missing_from_any_split)

if target_col in feature_cols:
    raise ValueError("Target variable is incorrectly included in predictors.")

print("\nTarget:", target_col)
print("Time variable:", time_col)
print("Series ID / predictor:", series_id_col)
print("Number of predictors:", len(feature_cols))
print("Predictors:", feature_cols)

# Clean final data

required_cols = [raw_id_col, series_id_col, time_col, target_col] + feature_cols

train_panel = train_panel.dropna(subset=required_cols).copy()
val_panel = val_panel.dropna(subset=required_cols).copy()
test_panel = test_panel.dropna(subset=required_cols).copy()

train_panel = train_panel.sort_values([series_id_col, time_col]).reset_index(drop=True)
val_panel = val_panel.sort_values([series_id_col, time_col]).reset_index(drop=True)
test_panel = test_panel.sort_values([series_id_col, time_col]).reset_index(drop=True)

# Window creation

def create_supervised_windows(data, context_length, feature_cols, target_col, series_id_col, time_col, raw_id_col):
    X_list = []
    y_list = []
    meta_list = []

    for gid, g in data.groupby(series_id_col, sort=True):
        g = g.sort_values(time_col).reset_index(drop=True)

        feats = g[feature_cols].to_numpy(dtype=np.float32)
        target = g[target_col].to_numpy(dtype=np.float32)
        raw_ids = g[raw_id_col].astype(str).to_numpy()
        dates = g[time_col].to_numpy()

        n = len(g)
        if n <= context_length:
            continue

        for i in range(context_length, n):
            x_window = feats[i - context_length:i].reshape(-1)
            y_value = target[i]

            if np.isnan(x_window).any() or np.isnan(y_value):
                continue

            X_list.append(x_window)
            y_list.append(y_value)
            meta_list.append({
                series_id_col: int(gid),
                raw_id_col: raw_ids[i],
                time_col: pd.Timestamp(dates[i]),
            })

    if len(X_list) == 0:
        return (
            np.empty((0, context_length * len(feature_cols)), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
            pd.DataFrame(columns=[series_id_col, raw_id_col, time_col]),
        )

    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.float32)
    meta = pd.DataFrame(meta_list)
    return X, y, meta


# Rolling one-step-ahead forecast

def rolling_one_step_forecast(history_df, future_df, context_length, feature_cols, target_col, series_id_col, time_col, raw_id_col, model):
    preds = []

    history_groups = {
        gid: g.sort_values(time_col).reset_index(drop=True).copy()
        for gid, g in history_df.groupby(series_id_col, sort=True)
    }
    future_groups = {
        gid: g.sort_values(time_col).reset_index(drop=True).copy()
        for gid, g in future_df.groupby(series_id_col, sort=True)
    }

    common_gids = sorted(set(history_groups.keys()) & set(future_groups.keys()))

    for gid in common_gids:
        hist = history_groups[gid]
        fut = future_groups[gid]

        if len(hist) < context_length:
            continue

        hist_features = hist[feature_cols].to_numpy(dtype=np.float32)

        for i in range(len(fut)):
            if hist_features.shape[0] < context_length:
                continue

            x_context = hist_features[-context_length:].reshape(1, -1)
            pred = float(model.predict(x_context)[0])

            current_row = fut.iloc[i]

            preds.append({
                series_id_col: int(gid),
                raw_id_col: str(current_row[raw_id_col]),
                time_col: current_row[time_col],
                "actual": float(current_row[target_col]),
                "predicted": pred,
            })

            new_feat_row = current_row[feature_cols].to_numpy(dtype=np.float32).reshape(1, -1)
            hist_features = np.vstack([hist_features, new_feat_row])

    return pd.DataFrame(preds)


# =========================
# Metrics
# =========================
def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mse = mean_squared_error(y_true, y_pred)

    return {
        "RMSE": float(np.sqrt(mse)),
        "MSE": float(mse),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 and np.var(y_true) > 0 else np.nan,
        "N": int(len(y_true)),
    }


# LightGBM factory

def build_lgbm_model(seed=42):
    return LGBMRegressor(
        objective="regression",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=0.0,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )


# Experiment loop

validation_results_all = []
test_results_all = []
window_counts = []

for context_length in CONTEXT_LENGTHS:
    print(f"\nRunning LightGBM with context length = {context_length}")

    X_train, y_train, train_meta = create_supervised_windows(
        data=train_panel,
        context_length=context_length,
        feature_cols=feature_cols,
        target_col=target_col,
        series_id_col=series_id_col,
        time_col=time_col,
        raw_id_col=raw_id_col,
    )

    print("Training windows:", len(X_train))

    if len(X_train) == 0:
        print("Skipping context length:", context_length)
        continue

    model = build_lgbm_model(seed=SEED)
    model.fit(X_train, y_train)

    print("Running rolling validation...")
    val_results = rolling_one_step_forecast(
        history_df=train_panel,
        future_df=val_panel,
        context_length=context_length,
        feature_cols=feature_cols,
        target_col=target_col,
        series_id_col=series_id_col,
        time_col=time_col,
        raw_id_col=raw_id_col,
        model=model,
    )

    if not val_results.empty:
        val_metrics = calculate_metrics(val_results["actual"], val_results["predicted"])
        validation_results_all.append({
            "Model": "LightGBM",
            "Split": "Validation",
            "ContextLength": context_length,
            **val_metrics,
        })
        print("Validation metrics:", val_metrics)

    train_val_history = (
        pd.concat([train_panel, val_panel], ignore_index=True)
        .sort_values([series_id_col, time_col])
        .reset_index(drop=True)
    )

    print("Running rolling test comparison...")
    test_results = rolling_one_step_forecast(
        history_df=train_val_history,
        future_df=test_panel,
        context_length=context_length,
        feature_cols=feature_cols,
        target_col=target_col,
        series_id_col=series_id_col,
        time_col=time_col,
        raw_id_col=raw_id_col,
        model=model,
    )

    if not test_results.empty:
        test_metrics = calculate_metrics(test_results["actual"], test_results["predicted"])
        test_results_all.append({
            "Model": "LightGBM",
            "Split": "Test_Comparison_Only",
            "ContextLength": context_length,
            **test_metrics,
        })
        print("Test comparison metrics:", test_metrics)

    window_counts.append({
        "ContextLength": context_length,
        "TrainWindows": len(X_train),
        "ValidationWindows": len(val_results),
        "ValidationGlaciers": val_results[series_id_col].nunique() if not val_results.empty else 0,
        "TestWindows": len(test_results),
        "TestGlaciers": test_results[series_id_col].nunique() if not test_results.empty else 0,
    })


# Validation selection

validation_metrics_df = pd.DataFrame(validation_results_all)
test_metrics_df = pd.DataFrame(test_results_all)
window_counts_df = pd.DataFrame(window_counts)

if not validation_metrics_df.empty:
    validation_metrics_df = validation_metrics_df.sort_values("RMSE").reset_index(drop=True)
    best_context = int(validation_metrics_df.iloc[0]["ContextLength"])
else:
    best_context = None

print("\nValidation comparison")
if not validation_metrics_df.empty:
    print(validation_metrics_df[["ContextLength", "RMSE", "MSE", "MAE", "R2", "N"]])

print("\nBest context based on validation RMSE:", best_context)

print("\nTest comparison results")
if not test_metrics_df.empty:
    test_metrics_df = test_metrics_df.sort_values("ContextLength").reset_index(drop=True)
    print(test_metrics_df[["ContextLength", "RMSE", "MSE", "MAE", "R2", "N"]])

print("\nWindow counts")
print(window_counts_df)


# Final model on train + validation

final_test_df = None

if best_context is not None:
    print("\nRetraining final LightGBM model...")
    print("Selected context:", best_context)

    train_val_df = (
        pd.concat([train_panel, val_panel], ignore_index=True)
        .sort_values([series_id_col, time_col])
        .reset_index(drop=True)
    )

    X_train_final, y_train_final, train_final_meta = create_supervised_windows(
        data=train_val_df,
        context_length=best_context,
        feature_cols=feature_cols,
        target_col=target_col,
        series_id_col=series_id_col,
        time_col=time_col,
        raw_id_col=raw_id_col,
    )

    print("Final training windows:", len(X_train_final))

    final_model = build_lgbm_model(seed=SEED)
    final_model.fit(X_train_final, y_train_final)

    print("Running official final test...")
    final_test_results = rolling_one_step_forecast(
        history_df=train_val_df,
        future_df=test_panel,
        context_length=best_context,
        feature_cols=feature_cols,
        target_col=target_col,
        series_id_col=series_id_col,
        time_col=time_col,
        raw_id_col=raw_id_col,
        model=final_model,
    )

    if not final_test_results.empty:
        final_test_metrics = calculate_metrics(
            final_test_results["actual"],
            final_test_results["predicted"]
        )

        final_test_df = pd.DataFrame([{
            "Model": "LightGBM",
            "Split": "Official_Final_Test",
            "ContextLength": best_context,
            **final_test_metrics,
        }])

        print("\nOfficial final test results")
        print(final_test_df[["ContextLength", "RMSE", "MSE", "MAE", "R2", "N"]])


print("\nLightGBM experiment complete")
print("Target:", target_col)
print("Series ID / predictor:", series_id_col)
print("Number of predictors:", len(feature_cols))
print("Context lengths:", CONTEXT_LENGTHS)
print("Prediction length:", PREDICTION_LENGTH)
print("Best context:", best_context)

In [ ]:
final_test_results_all = []

for context_length in [12, 24, 36]:
    print(f"\nRetraining final model for context = {context_length}")

    train_val_df = (
        pd.concat([train_panel, val_panel], ignore_index=True)
        .sort_values([series_id_col, time_col])
        .reset_index(drop=True)
    )

    X_train_final, y_train_final, _ = create_supervised_windows(
        data=train_val_df,
        context_length=context_length,
        feature_cols=feature_cols,
        target_col=target_col,
        series_id_col=series_id_col,
        time_col=time_col,
        raw_id_col=raw_id_col,
    )

    if len(X_train_final) == 0:
        print("Skipping context:", context_length)
        continue

    final_model = build_lgbm_model(seed=SEED)
    final_model.fit(X_train_final, y_train_final)

    final_test_results = rolling_one_step_forecast(
        history_df=train_val_df,
        future_df=test_panel,
        context_length=context_length,
        feature_cols=feature_cols,
        target_col=target_col,
        series_id_col=series_id_col,
        time_col=time_col,
        raw_id_col=raw_id_col,
        model=final_model,
    )

    if not final_test_results.empty:
        final_test_metrics = calculate_metrics(
            final_test_results["actual"],
            final_test_results["predicted"]
        )

        final_test_results_all.append({
            "Model": "LightGBM",
            "Split": "Official_Final_Test",
            "ContextLength": context_length,
            **final_test_metrics,
        })

final_test_metrics_df = pd.DataFrame(final_test_results_all)
print(final_test_metrics_df[["ContextLength", "RMSE", "MSE", "MAE", "R2", "N"]])

In [ ]:
# random
import os
import warnings
import random

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


SEED = 42

random.seed(SEED)
np.random.seed(SEED)


DATA_DIR = "/home/parcot1/updated_data"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
VALIDATION_PATH = os.path.join(DATA_DIR, "validation.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

OUTPUT_DIR = os.path.join(
    DATA_DIR,
    "RandomForest_results"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


TARGET_COL = "retreat_change_next_month"

GLACIER_COL = "glacier"

GLACIER_CODE_COL = "glacier_code"

TIME_COL = "datetime"


CONTEXT_LENGTHS = [
    6,
    12,
    24,
    36
]


PREDICTOR_COLS = [
    "glacier_code",
    "time_idx",
    "year",
    "month",
    "month_sin",
    "month_cos",
    "retreat",
    "retreat_change",
    "retreat_lag_1",
    "retreat_change_lag_1",
    "terminus_thermal",
    "shelf_thermal",
    "undercutting",
    "discharge",
    "x_epsg3413",
    "y_epsg3413",
    "basin_CE",
    "basin_CW",
    "basin_N",
    "basin_NE",
    "basin_NW",
    "basin_SE",
    "basin_SW",
    "category_CR",
    "category_DW",
    "category_FE",
    "category_NC",
    "category_SC",
    "category_SR"
]


RANDOM_FOREST_PARAMS = {
    "n_estimators": 100,
    "max_depth": 15,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "random_state": SEED,
    "n_jobs": -1
}


def calculate_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mse = mean_squared_error(
        y_true,
        y_pred
    )

    if (
        len(y_true) > 1
        and np.var(y_true) > 0
    ):
        r2 = r2_score(
            y_true,
            y_pred
        )
    else:
        r2 = np.nan

    return {
        "RMSE": float(
            np.sqrt(mse)
        ),
        "MSE": float(
            mse
        ),
        "MAE": float(
            mean_absolute_error(
                y_true,
                y_pred
            )
        ),
        "R2": float(
            r2
        ),
        "N": int(
            len(y_true)
        )
    }


def load_data(
    path
):

    df = pd.read_csv(
        path
    )

    df[TIME_COL] = pd.to_datetime(
        df[TIME_COL],
        errors="coerce"
    )

    df[GLACIER_COL] = (
        df[GLACIER_COL]
        .astype(str)
        .str.strip()
    )

    df[TARGET_COL] = pd.to_numeric(
        df[TARGET_COL],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            TIME_COL,
            GLACIER_COL,
            TARGET_COL
        ]
    ).copy()

    df = df.sort_values(
        [
            GLACIER_COL,
            TIME_COL
        ]
    ).reset_index(
        drop=True
    )

    return df


print("Loading data...")

train_df = load_data(
    TRAIN_PATH
)

val_df = load_data(
    VALIDATION_PATH
)

test_df = load_data(
    TEST_PATH
)

print(
    "Train shape:",
    train_df.shape
)

print(
    "Validation shape:",
    val_df.shape
)

print(
    "Test shape:",
    test_df.shape
)


train_ids = set(
    train_df[
        GLACIER_COL
    ].unique()
)

val_ids = set(
    val_df[
        GLACIER_COL
    ].unique()
)

test_ids = set(
    test_df[
        GLACIER_COL
    ].unique()
)


common_ids = (
    train_ids
    & val_ids
    & test_ids
)


print(
    "Common glaciers:",
    len(common_ids)
)


train_df = train_df[
    train_df[
        GLACIER_COL
    ].isin(
        common_ids
    )
].copy()

val_df = val_df[
    val_df[
        GLACIER_COL
    ].isin(
        common_ids
    )
].copy()

test_df = test_df[
    test_df[
        GLACIER_COL
    ].isin(
        common_ids
    )
].copy()


all_common_glaciers = sorted(
    pd.concat(
        [
            train_df[
                [GLACIER_COL]
            ],
            val_df[
                [GLACIER_COL]
            ],
            test_df[
                [GLACIER_COL]
            ]
        ],
        ignore_index=True
    )[
        GLACIER_COL
    ]
    .dropna()
    .astype(str)
    .unique()
)


glacier_mapping = {
    glacier: code
    for code, glacier in enumerate(
        all_common_glaciers
    )
}


def add_glacier_code(
    df
):

    df = df.copy()

    df[
        GLACIER_CODE_COL
    ] = (
        df[
            GLACIER_COL
        ].map(
            glacier_mapping
        )
    )

    if df[
        GLACIER_CODE_COL
    ].isna().any():

        unknown_glaciers = (
            df.loc[
                df[
                    GLACIER_CODE_COL
                ].isna(),
                GLACIER_COL
            ]
            .unique()
        )

        raise ValueError(
            f"Unknown glacier IDs: "
            f"{unknown_glaciers}"
        )

    df[
        GLACIER_CODE_COL
    ] = (
        df[
            GLACIER_CODE_COL
        ].astype(
            np.int32
        )
    )

    return df


train_df = add_glacier_code(
    train_df
)

val_df = add_glacier_code(
    val_df
)

test_df = add_glacier_code(
    test_df
)


for df in [
    train_df,
    val_df,
    test_df
]:

    for col in PREDICTOR_COLS:

        if col in df.columns:

            df[
                col
            ] = pd.to_numeric(
                df[
                    col
                ],
                errors="coerce"
            )


required_columns = list(
    dict.fromkeys(
        [
            GLACIER_COL,
            GLACIER_CODE_COL,
            TIME_COL,
            TARGET_COL
        ]
        + PREDICTOR_COLS
    )
)


for name, df in [
    (
        "Training",
        train_df
    ),
    (
        "Validation",
        val_df
    ),
    (
        "Test",
        test_df
    )
]:

    missing_columns = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"{name} data is missing "
            f"columns: "
            f"{missing_columns}"
        )


train_df = train_df.dropna(
    subset=required_columns
).copy()

val_df = val_df.dropna(
    subset=required_columns
).copy()

test_df = test_df.dropna(
    subset=required_columns
).copy()


train_df = train_df.sort_values(
    [
        GLACIER_COL,
        TIME_COL
    ]
).reset_index(
    drop=True
)

val_df = val_df.sort_values(
    [
        GLACIER_COL,
        TIME_COL
    ]
).reset_index(
    drop=True
)

test_df = test_df.sort_values(
    [
        GLACIER_COL,
        TIME_COL
    ]
).reset_index(
    drop=True
)


validation_dates = sorted(
    val_df[
        TIME_COL
    ].drop_duplicates().tolist()
)

test_dates = sorted(
    test_df[
        TIME_COL
    ].drop_duplicates().tolist()
)


print(
    "Number of predictors:",
    len(PREDICTOR_COLS)
)

print(
    "Predictors:"
)

for col in PREDICTOR_COLS:

    print(
        " ",
        col
    )


print(
    "\nRandom Forest parameters:"
)

print(
    RANDOM_FOREST_PARAMS
)


def run_rolling_predictions(
    train_history_df,
    future_df,
    prediction_dates,
    context_length,
    split_name
):

    all_results = []

    history_df = (
        train_history_df
        .copy()
    )

    future_df = (
        future_df
        .copy()
    )

    history_df = (
        history_df
        .sort_values(
            [
                GLACIER_COL,
                TIME_COL
            ]
        )
        .reset_index(
            drop=True
        )
    )

    future_df = (
        future_df
        .sort_values(
            [
                GLACIER_COL,
                TIME_COL
            ]
        )
        .reset_index(
            drop=True
        )
    )

    for prediction_date in prediction_dates:

        history_cut = history_df[
            history_df[
                TIME_COL
            ] < prediction_date
        ].copy()

        prediction_slice = future_df[
            future_df[
                TIME_COL
            ] == prediction_date
        ].copy()

        if (
            history_cut.empty
            or prediction_slice.empty
        ):

            continue


        history_cut = (
            history_cut
            .sort_values(
                [
                    GLACIER_COL,
                    TIME_COL
                ]
            )
            .groupby(
                GLACIER_COL,
                group_keys=False
            )
            .tail(
                context_length
            )
            .copy()
        )


        if history_cut.empty:

            continue


        X_train = (
            history_cut[
                PREDICTOR_COLS
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        y_train = (
            history_cut[
                TARGET_COL
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        X_pred = (
            prediction_slice[
                PREDICTOR_COLS
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        model = RandomForestRegressor(
            **RANDOM_FOREST_PARAMS
        )


        model.fit(
            X_train,
            y_train
        )


        y_pred = model.predict(
            X_pred
        )


        result_df = prediction_slice[
            [
                GLACIER_COL,
                GLACIER_CODE_COL,
                TIME_COL,
                TARGET_COL
            ]
        ].copy()


        result_df = result_df.rename(
            columns={
                TARGET_COL: "actual"
            }
        )


        result_df[
            "predicted"
        ] = y_pred


        result_df[
            "Split"
        ] = split_name


        result_df[
            "ContextLength"
        ] = context_length


        all_results.append(
            result_df
        )


        revealed_rows = prediction_slice.copy()


        history_df = pd.concat(
            [
                history_df,
                revealed_rows
            ],
            ignore_index=True
        )


        history_df = (
            history_df
            .sort_values(
                [
                    GLACIER_COL,
                    TIME_COL
                ]
            )
            .reset_index(
                drop=True
            )
        )


    if not all_results:

        return pd.DataFrame()


    return (
        pd.concat(
            all_results,
            ignore_index=True
        )
        .sort_values(
            [
                GLACIER_COL,
                TIME_COL
            ]
        )
        .reset_index(
            drop=True
        )
    )


all_metrics = []

all_test_results = []


for context_length in CONTEXT_LENGTHS:

    print(
        "\n"
        + "=" * 60
    )

    print(
        "Context Length:",
        context_length
    )

    print(
        "=" * 60
    )


    print(
        "Running validation..."
    )


    validation_results = (
        run_rolling_predictions(
            train_history_df=train_df,
            future_df=val_df,
            prediction_dates=validation_dates,
            context_length=context_length,
            split_name="Validation"
        )
    )


    if not validation_results.empty:

        validation_metrics = (
            calculate_metrics(
                validation_results[
                    "actual"
                ],
                validation_results[
                    "predicted"
                ]
            )
        )


        all_metrics.append(
            {
                "Model": "RandomForest",
                "Split": "Validation",
                "ContextLength": context_length,
                **validation_metrics
            }
        )


        print(
            "Validation RMSE:",
            round(
                validation_metrics[
                    "RMSE"
                ],
                6
            )
        )


        validation_results.to_csv(
            os.path.join(
                OUTPUT_DIR,
                f"RandomForest_validation_context_{context_length}.csv"
            ),
            index=False
        )


    print(
        "Running test..."
    )


    train_val_history = pd.concat(
        [
            train_df,
            val_df
        ],
        ignore_index=True
    )


    train_val_history = (
        train_val_history
        .sort_values(
            [
                GLACIER_COL,
                TIME_COL
            ]
        )
        .reset_index(
            drop=True
        )
    )


    test_results = (
        run_rolling_predictions(
            train_history_df=train_val_history,
            future_df=test_df,
            prediction_dates=test_dates,
            context_length=context_length,
            split_name="Test"
        )
    )


    if not test_results.empty:

        test_metrics = (
            calculate_metrics(
                test_results[
                    "actual"
                ],
                test_results[
                    "predicted"
                ]
            )
        )


        all_metrics.append(
            {
                "Model": "RandomForest",
                "Split": "Test",
                "ContextLength": context_length,
                **test_metrics
            }
        )


        all_test_results.append(
            test_results
        )


        print(
            "Test RMSE:",
            round(
                test_metrics[
                    "RMSE"
                ],
                6
            )
        )


        test_results.to_csv(
            os.path.join(
                OUTPUT_DIR,
                f"RandomForest_test_context_{context_length}.csv"
            ),
            index=False
        )


metrics_df = pd.DataFrame(
    all_metrics
)


if not metrics_df.empty:

    metrics_df = (
        metrics_df
        .sort_values(
            [
                "Split",
                "ContextLength"
            ]
        )
        .reset_index(
            drop=True
        )
    )


metrics_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "RandomForest_all_metrics.csv"
    ),
    index=False
)


if all_test_results:

    combined_test_results = pd.concat(
        all_test_results,
        ignore_index=True
    )

    combined_test_results.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "RandomForest_all_test_predictions.csv"
        ),
        index=False
    )

else:

    combined_test_results = (
        pd.DataFrame()
    )


print(
    "\nRandom Forest Results"
)

if not metrics_df.empty:

    print(
        metrics_df.to_string(
            index=False
        )
    )

else:

    print(
        "No results generated."
    )


print(
    "\nResults saved to:"
)

print(
    OUTPUT_DIR
)


print(
    "\nRandom Forest experiment complete."
)

In [ ]:
# lstm
import os
import warnings
import random

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


DATA_DIR = "/home/parcot1/updated_data"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
VALIDATION_PATH = os.path.join(DATA_DIR, "validation.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

OUTPUT_DIR = os.path.join(DATA_DIR, "LSTM_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)


TARGET_COL = "retreat_change_next_month"
GLACIER_COL = "glacier"
GLACIER_CODE_COL = "glacier_code"
TIME_COL = "datetime"


CONTEXT_LENGTHS = [6, 12, 24, 36]

SEQUENCE_LENGTH = 1

HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2

BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 0.001

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


PREDICTOR_COLS = [
    "glacier_code",
    "time_idx",
    "year",
    "month",
    "month_sin",
    "month_cos",
    "retreat",
    "retreat_change",
    "retreat_lag_1",
    "retreat_change_lag_1",
    "terminus_thermal",
    "shelf_thermal",
    "undercutting",
    "discharge",
    "x_epsg3413",
    "y_epsg3413",
    "basin_CE",
    "basin_CW",
    "basin_N",
    "basin_NE",
    "basin_NW",
    "basin_SE",
    "basin_SW",
    "category_CR",
    "category_DW",
    "category_FE",
    "category_NC",
    "category_SC",
    "category_SR"
]


print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mse = mean_squared_error(
        y_true,
        y_pred
    )

    if (
        len(y_true) > 1
        and np.var(y_true) > 0
    ):
        r2 = r2_score(
            y_true,
            y_pred
        )
    else:
        r2 = np.nan

    return {
        "RMSE": float(
            np.sqrt(mse)
        ),
        "MSE": float(mse),
        "MAE": float(
            mean_absolute_error(
                y_true,
                y_pred
            )
        ),
        "R2": float(r2),
        "N": int(
            len(y_true)
        )
    }


def load_data(path):

    df = pd.read_csv(path)

    df[TIME_COL] = pd.to_datetime(
        df[TIME_COL],
        errors="coerce"
    )

    df[GLACIER_COL] = (
        df[GLACIER_COL]
        .astype(str)
        .str.strip()
    )

    df[TARGET_COL] = pd.to_numeric(
        df[TARGET_COL],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            TIME_COL,
            GLACIER_COL,
            TARGET_COL
        ]
    ).copy()

    df = df.sort_values(
        [
            GLACIER_COL,
            TIME_COL
        ]
    ).reset_index(
        drop=True
    )

    return df


print("Loading data...")

train_df = load_data(
    TRAIN_PATH
)

val_df = load_data(
    VALIDATION_PATH
)

test_df = load_data(
    TEST_PATH
)

print(
    "Train shape:",
    train_df.shape
)

print(
    "Validation shape:",
    val_df.shape
)

print(
    "Test shape:",
    test_df.shape
)


train_ids = set(
    train_df[
        GLACIER_COL
    ].unique()
)

val_ids = set(
    val_df[
        GLACIER_COL
    ].unique()
)

test_ids = set(
    test_df[
        GLACIER_COL
    ].unique()
)


common_ids = (
    train_ids
    & val_ids
    & test_ids
)


print(
    "Common glaciers:",
    len(common_ids)
)


train_df = train_df[
    train_df[
        GLACIER_COL
    ].isin(common_ids)
].copy()

val_df = val_df[
    val_df[
        GLACIER_COL
    ].isin(common_ids)
].copy()

test_df = test_df[
    test_df[
        GLACIER_COL
    ].isin(common_ids)
].copy()


all_common_glaciers = sorted(
    pd.concat(
        [
            train_df[
                [GLACIER_COL]
            ],
            val_df[
                [GLACIER_COL]
            ],
            test_df[
                [GLACIER_COL]
            ]
        ],
        ignore_index=True
    )[
        GLACIER_COL
    ]
    .dropna()
    .astype(str)
    .unique()
)


glacier_mapping = {
    glacier: code
    for code, glacier in enumerate(
        all_common_glaciers
    )
}


def add_glacier_code(df):

    df = df.copy()

    df[
        GLACIER_CODE_COL
    ] = (
        df[
            GLACIER_COL
        ]
        .map(
            glacier_mapping
        )
    )

    if df[
        GLACIER_CODE_COL
    ].isna().any():

        unknown_glaciers = (
            df.loc[
                df[
                    GLACIER_CODE_COL
                ].isna(),
                GLACIER_COL
            ]
            .unique()
        )

        raise ValueError(
            f"Unknown glaciers: "
            f"{unknown_glaciers}"
        )

    df[
        GLACIER_CODE_COL
    ] = (
        df[
            GLACIER_CODE_COL
        ]
        .astype(
            np.int32
        )
    )

    return df


train_df = add_glacier_code(
    train_df
)

val_df = add_glacier_code(
    val_df
)

test_df = add_glacier_code(
    test_df
)


for df in [
    train_df,
    val_df,
    test_df
]:

    for col in PREDICTOR_COLS:

        if col in df.columns:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )


required_columns = (
    [
        GLACIER_COL,
        GLACIER_CODE_COL,
        TIME_COL,
        TARGET_COL
    ]
    + PREDICTOR_COLS
)


for name, df in [
    (
        "Training",
        train_df
    ),
    (
        "Validation",
        val_df
    ),
    (
        "Test",
        test_df
    )
]:

    missing_columns = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"{name} data is missing "
            f"columns: "
            f"{missing_columns}"
        )


train_df = train_df.dropna(
    subset=required_columns
).copy()

val_df = val_df.dropna(
    subset=required_columns
).copy()

test_df = test_df.dropna(
    subset=required_columns
).copy()


train_df = train_df.sort_values(
    [
        GLACIER_COL,
        TIME_COL
    ]
).reset_index(
    drop=True
)

val_df = val_df.sort_values(
    [
        GLACIER_COL,
        TIME_COL
    ]
).reset_index(
    drop=True
)

test_df = test_df.sort_values(
    [
        GLACIER_COL,
        TIME_COL
    ]
).reset_index(
    drop=True
)


print(
    "Number of predictors:",
    len(PREDICTOR_COLS)
)

print(
    "Predictors:"
)

for col in PREDICTOR_COLS:

    print(
        " ",
        col
    )


class LSTMRegressor(
    nn.Module
):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        dropout
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=(
                dropout
                if num_layers > 1
                else 0.0
            )
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )


    def forward(
        self,
        x
    ):

        output, _ = self.lstm(
            x
        )

        last_output = (
            output[:, -1, :]
        )

        prediction = self.fc(
            last_output
        )

        return prediction.squeeze(
            -1
        )


def create_training_windows(
    df,
    context_length,
    feature_scaler,
    target_scaler
):

    X_windows = []

    y_windows = []

    for glacier, glacier_df in df.groupby(
        GLACIER_COL
    ):

        glacier_df = (
            glacier_df
            .sort_values(
                TIME_COL
            )
            .reset_index(
                drop=True
            )
        )

        if len(
            glacier_df
        ) <= context_length:

            continue


        features = (
            glacier_df[
                PREDICTOR_COLS
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        targets = (
            glacier_df[
                TARGET_COL
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        scaled_features = (
            feature_scaler.transform(
                features
            )
        )

        scaled_targets = (
            target_scaler.transform(
                targets.reshape(
                    -1,
                    1
                )
            )
            .flatten()
        )


        for i in range(
            context_length,
            len(glacier_df)
        ):

            X_windows.append(
                scaled_features[
                    i - context_length:i
                ]
            )

            y_windows.append(
                scaled_targets[
                    i
                ]
            )


    if not X_windows:

        return (
            np.empty(
                (
                    0,
                    context_length,
                    len(
                        PREDICTOR_COLS
                    )
                ),
                dtype=np.float32
            ),
            np.empty(
                (
                    0,
                ),
                dtype=np.float32
            )
        )


    return (
        np.asarray(
            X_windows,
            dtype=np.float32
        ),
        np.asarray(
            y_windows,
            dtype=np.float32
        )
    )


def fit_scalers(
    df
):

    feature_scaler = StandardScaler()

    target_scaler = StandardScaler()


    all_features = (
        df[
            PREDICTOR_COLS
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    all_targets = (
        df[
            TARGET_COL
        ]
        .to_numpy(
            dtype=np.float32
        )
        .reshape(
            -1,
            1
        )
    )


    feature_scaler.fit(
        all_features
    )

    target_scaler.fit(
        all_targets
    )


    return (
        feature_scaler,
        target_scaler
    )


def train_lstm(
    train_data,
    context_length
):

    feature_scaler, target_scaler = (
        fit_scalers(
            train_data
        )
    )


    X_train, y_train = (
        create_training_windows(
            train_data,
            context_length,
            feature_scaler,
            target_scaler
        )
    )


    if len(
        X_train
    ) == 0:

        raise ValueError(
            "No training windows "
            "were created."
        )


    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.float32
    )


    dataset = torch.utils.data.TensorDataset(
        X_train_tensor,
        y_train_tensor
    )


    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )


    model = LSTMRegressor(
        input_size=len(
            PREDICTOR_COLS
        ),
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    ).to(
        DEVICE
    )


    criterion = nn.MSELoss()


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )


    model.train()


    for epoch in range(
        EPOCHS
    ):

        epoch_loss = 0.0


        for X_batch, y_batch in loader:

            X_batch = X_batch.to(
                DEVICE
            )

            y_batch = y_batch.to(
                DEVICE
            )


            optimizer.zero_grad()


            predictions = model(
                X_batch
            )


            loss = criterion(
                predictions,
                y_batch
            )


            loss.backward()


            optimizer.step()


            epoch_loss += (
                loss.item()
                * len(
                    X_batch
                )
            )


        epoch_loss /= len(
            dataset
        )


        if (
            epoch == 0
            or (
                epoch + 1
            ) % 10 == 0
        ):

            print(
                f"Epoch "
                f"{epoch + 1}/{EPOCHS} "
                f"- Loss: "
                f"{epoch_loss:.6f}"
            )


    return (
        model,
        feature_scaler,
        target_scaler
    )


def predict_one_step(
    model,
    history_df,
    prediction_row,
    context_length,
    feature_scaler,
    target_scaler
):

    glacier = prediction_row[
        GLACIER_COL
    ]


    glacier_history = (
        history_df[
            history_df[
                GLACIER_COL
            ] == glacier
        ]
        .sort_values(
            TIME_COL
        )
        .drop_duplicates(
            subset=[
                TIME_COL
            ],
            keep="last"
        )
    )


    if len(
        glacier_history
    ) < context_length:

        return None


    context = (
        glacier_history
        .tail(
            context_length
        )
    )


    features = (
        context[
            PREDICTOR_COLS
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    features = (
        feature_scaler.transform(
            features
        )
    )


    X = torch.tensor(
        features,
        dtype=torch.float32
    ).unsqueeze(
        0
    ).to(
        DEVICE
    )


    model.eval()


    with torch.no_grad():

        prediction_scaled = (
            model(
                X
            )
            .cpu()
            .numpy()
            .reshape(
                -1,
                1
            )
        )


    prediction = (
        target_scaler.inverse_transform(
            prediction_scaled
        )
        .flatten()[0]
    )


    return float(
        prediction
    )


def rolling_forecast(
    model,
    initial_history,
    forecast_df,
    context_length,
    feature_scaler,
    target_scaler,
    split_name
):

    history = (
        initial_history
        .copy()
        .sort_values(
            [
                GLACIER_COL,
                TIME_COL
            ]
        )
        .reset_index(
            drop=True
        )
    )


    forecast_df = (
        forecast_df
        .copy()
        .sort_values(
            [
                TIME_COL,
                GLACIER_COL
            ]
        )
        .reset_index(
            drop=True
        )
    )


    results = []


    for _, row in forecast_df.iterrows():

        prediction = predict_one_step(
            model=model,
            history_df=history,
            prediction_row=row,
            context_length=context_length,
            feature_scaler=feature_scaler,
            target_scaler=target_scaler
        )


        if prediction is None:

            continue


        results.append(
            {
                GLACIER_COL:
                    row[
                        GLACIER_COL
                    ],

                GLACIER_CODE_COL:
                    int(
                        row[
                            GLACIER_CODE_COL
                        ]
                    ),

                TIME_COL:
                    row[
                        TIME_COL
                    ],

                "actual":
                    float(
                        row[
                            TARGET_COL
                        ]
                    ),

                "predicted":
                    float(
                        prediction
                    ),

                "Split":
                    split_name,

                "ContextLength":
                    context_length
            }
        )


        history = pd.concat(
            [
                history,
                pd.DataFrame(
                    [row]
                )
            ],
            ignore_index=True
        )


        history = (
            history
            .drop_duplicates(
                subset=[
                    GLACIER_COL,
                    TIME_COL
                ],
                keep="last"
            )
            .sort_values(
                [
                    GLACIER_COL,
                    TIME_COL
                ]
            )
            .reset_index(
                drop=True
            )
        )


    return pd.DataFrame(
        results
    )


all_metrics = []


for context_length in CONTEXT_LENGTHS:

    print(
        "\n"
        + "=" * 60
    )

    print(
        "LSTM Context Length:",
        context_length
    )

    print(
        "=" * 60
    )


    print(
        "Training LSTM..."
    )


    model, feature_scaler, target_scaler = (
        train_lstm(
            train_df,
            context_length
        )
    )


    print(
        "Running validation..."
    )


    validation_results = (
        rolling_forecast(
            model=model,
            initial_history=train_df,
            forecast_df=val_df,
            context_length=context_length,
            feature_scaler=feature_scaler,
            target_scaler=target_scaler,
            split_name="Validation"
        )
    )


    if not validation_results.empty:

        validation_metrics = (
            calculate_metrics(
                validation_results[
                    "actual"
                ],
                validation_results[
                    "predicted"
                ]
            )
        )


        all_metrics.append(
            {
                "Model": "LSTM",
                "Split": "Validation",
                "ContextLength":
                    context_length,
                **validation_metrics
            }
        )


        print(
            "Validation RMSE:",
            round(
                validation_metrics[
                    "RMSE"
                ],
                6
            )
        )


        validation_results.to_csv(
            os.path.join(
                OUTPUT_DIR,
                f"LSTM_validation_context_{context_length}.csv"
            ),
            index=False
        )


    print(
        "Running test..."
    )


    train_val_history = pd.concat(
        [
            train_df,
            val_df
        ],
        ignore_index=True
    )


    train_val_history = (
        train_val_history
        .drop_duplicates(
            subset=[
                GLACIER_COL,
                TIME_COL
            ],
            keep="last"
        )
        .sort_values(
            [
                GLACIER_COL,
                TIME_COL
            ]
        )
        .reset_index(
            drop=True
        )
    )


    test_results = (
        rolling_forecast(
            model=model,
            initial_history=train_val_history,
            forecast_df=test_df,
            context_length=context_length,
            feature_scaler=feature_scaler,
            target_scaler=target_scaler,
            split_name="Test"
        )
    )


    if not test_results.empty:

        test_metrics = (
            calculate_metrics(
                test_results[
                    "actual"
                ],
                test_results[
                    "predicted"
                ]
            )
        )


        all_metrics.append(
            {
                "Model": "LSTM",
                "Split": "Test",
                "ContextLength":
                    context_length,
                **test_metrics
            }
        )


        print(
            "Test RMSE:",
            round(
                test_metrics[
                    "RMSE"
                ],
                6
            )
        )


        test_results.to_csv(
            os.path.join(
                OUTPUT_DIR,
                f"LSTM_test_context_{context_length}.csv"
            ),
            index=False
        )


    del model


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


metrics_df = pd.DataFrame(
    all_metrics
)


metrics_df = (
    metrics_df
    .sort_values(
        [
            "Split",
            "ContextLength"
        ]
    )
    .reset_index(
        drop=True
    )
)


metrics_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "LSTM_all_metrics.csv"
    ),
    index=False
)


print(
    "\nLSTM Results"
)

print(
    metrics_df.to_string(
        index=False
    )
)


print(
    "\nResults saved to:"
)

print(
    OUTPUT_DIR
)


print(
    "\nLSTM experiment complete."
)

In [ ]:
# moirai Finetuned

import os
import math
import random
import warnings

warnings.filterwarnings("ignore")


import numpy as np
import pandas as pd
import torch

from IPython.display import display

from torch.utils.data import Dataset, DataLoader


from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


from uni2ts.model.moirai import (
    MoiraiModule,
    MoiraiForecast
)


from uni2ts.model.moirai.finetune import (
    MoiraiFinetune
)


from uni2ts.loss.packed import (
    PackedNLLLoss
)



#  Configuration

MODEL_NAME = "Salesforce/moirai-1.1-R-small"


DEVICE = torch.device("cpu")


DATA_DIR = "/home/parcot1/imputed_data"


TRAIN_PATH = os.path.join(
    DATA_DIR,
    "D3_train_encoded.csv"
)


VAL_PATH = os.path.join(
    DATA_DIR,
    "D3_validation_encoded.csv"
)


TEST_PATH = os.path.join(
    DATA_DIR,
    "D3_test_encoded.csv"
)


# Columns

SERIES_ID_COL = "glacier"


TARGET_COL = "retreat_change_next_month"



PREDICTOR_COLS = [

    "time_idx",

    "retreat",

    "retreat_change",

    "retreat_lag_1",

    "retreat_change_lag_1",

    "terminus_thermal",

    "shelf_thermal",

    "undercutting",

    "discharge",


    # calendar
    "year",

    "month_num",

    "month_sin",

    "month_cos",


    # basin
    "basin_CE",
    "basin_CW",
    "basin_N",
    "basin_NE",
    "basin_NW",
    "basin_SE",
    "basin_SW",


    # glacier category
    "category_CR",
    "category_DW",
    "category_FE",
    "category_NC",
    "category_SC",
    "category_SR",


    "glacier_code"

]

# Moirai parameters

CONTEXTS = [
    6,
    12,
    24,
    36
]


PREDICTION_LENGTH = 1


PATCH_SIZE = 16


MAX_PATCH_SIZE = 128


NUM_VARIATES = len(PREDICTOR_COLS) + 1


TARGET_VARIATE_INDEX = NUM_VARIATES - 1


BATCH_SIZE = 32


EPOCHS = 10


LEARNING_RATE = 1e-5


WEIGHT_DECAY = 1e-5


NUM_SAMPLES = 100


NUM_WORKERS = 0


SEED = 42



MAX_TRAIN_BATCHES_PER_EPOCH = 100

MAX_VAL_BATCHES = 50

MAX_TEST_BATCHES = None



# Seed

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)



print("="*70)
print("MOIRAI-1.1-R-SMALL FINE-TUNING")
print("="*70)


print("Model:", MODEL_NAME)

print("Device:", DEVICE)

print("Contexts:", CONTEXTS)

print("Predictors:", len(PREDICTOR_COLS))

print("Total variates:", NUM_VARIATES)

print("Target:", TARGET_COL)



# File checking

for path in [
    TRAIN_PATH,
    VAL_PATH,
    TEST_PATH
]:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Missing file:\n{path}"
        )


# Load datasets

train_df = pd.read_csv(TRAIN_PATH)

val_df = pd.read_csv(VAL_PATH)

test_df = pd.read_csv(TEST_PATH)



print()

print("DATA LOADED")

print("Train:", train_df.shape)

print("Validation:", val_df.shape)

print("Test:", test_df.shape)



#  Datetime

for df in [
    train_df,
    val_df,
    test_df
]:

    df["datetime"] = pd.to_datetime(
        df["datetime"]
    )



# Glacier encoding
print()

print("CREATING GLACIER CODE")


glaciers = sorted(
    train_df[SERIES_ID_COL]
    .astype(str)
    .unique()
)



glacier_to_code = {

    g:i

    for i,g in enumerate(glaciers)

}



print(
    "Number of glaciers:",
    len(glacier_to_code)
)



for name, df in [

    ("TRAIN", train_df),

    ("VALIDATION", val_df),

    ("TEST", test_df)

]:

    df["glacier"] = (
        df["glacier"]
        .astype(str)
    )


    df["glacier_code"] = (
        df["glacier"]
        .map(glacier_to_code)
    )


    if df["glacier_code"].isna().any():

        raise ValueError(
            f"{name} contains unseen glaciers"
        )


    df["glacier_code"] = (
        df["glacier_code"]
        .astype(np.float32)
    )



# Calendar features

print()

print("CREATING CALENDAR FEATURES")



for df in [

    train_df,

    val_df,

    test_df

]:

    # year
    df["year"] = (
        df["datetime"]
        .dt.year
        .astype(np.float32)
    )


    # month number 1-12
    df["month_num"] = (
        df["datetime"]
        .dt.month
        .astype(np.float32)
    )


    # cyclic encoding

    df["month_sin"] = (

        np.sin(
            2*np.pi*
            (df["month_num"]-1)
            /12
        )

    ).astype(np.float32)



    df["month_cos"] = (

        np.cos(
            2*np.pi*
            (df["month_num"]-1)
            /12
        )

    ).astype(np.float32)


# Verify columns
required_columns = (
    PREDICTOR_COLS
    +
    [TARGET_COL]
)



print()

print("VERIFYING DATASETS")



for name, df in [

    ("TRAIN", train_df),

    ("VALIDATION", val_df),

    ("TEST", test_df)

]:


    missing = [

        c for c in required_columns

        if c not in df.columns

    ]


    if missing:

        raise ValueError(
            f"{name} missing columns: {missing}"
        )


    nan_count = (
        df[required_columns]
        .isna()
        .sum()
        .sum()
    )


    if nan_count > 0:

        raise ValueError(
            f"{name} contains NaN values"
        )


    print(
        name,
        "OK",
        df.shape
    )



print()

print("PART 1 COMPLETED SUCCESSFULLY")

print("BUILDING HISTORICAL DATA")

train_df["_split"] = "train"
val_df["_split"] = "validation"
test_df["_split"] = "test"


all_df = pd.concat(
    [
        train_df,
        val_df,
        test_df
    ],
    axis=0,
    ignore_index=True
)


all_df = all_df.sort_values(
    [
        SERIES_ID_COL,
        "datetime"
    ]
).reset_index(drop=True)



print()

print("Total rows:", len(all_df))

print(
    "Unique glaciers:",
    all_df[SERIES_ID_COL].nunique()
)



# Verify time ordering

for glacier, group in all_df.groupby(SERIES_ID_COL):

    group = group.sort_values("datetime")

    if not group["datetime"].is_monotonic_increasing:

        raise ValueError(
            f"Datetime ordering problem: {glacier}"
        )


print("Historical data sorted")

# Glacier Window Dataset

class GlacierWindowDataset(Dataset):


    def __init__(
        self,
        dataframe,
        context_length,
        prediction_length,
        target_split
    ):


        self.context_length = context_length

        self.prediction_length = prediction_length

        self.target_split = target_split

        self.samples = []



        for glacier, group in dataframe.groupby(
            SERIES_ID_COL
        ):


            group = (
                group
                .sort_values("datetime")
                .reset_index(drop=True)
            )



            values = (

                group[
                    PREDICTOR_COLS +
                    [TARGET_COL]
                ]

                .astype(np.float32)
                .values

            )



            split_values = (
                group["_split"]
                .values
            )



            dates = (
                group["datetime"]
                .values
            )



            for i in range(
                context_length,
                len(group)-prediction_length+1
            ):


                if split_values[i] != target_split:

                    continue



                context_values = (

                    values[
                        i-context_length:i
                    ]

                )


                future_values = (

                    values[
                        i:i+prediction_length
                    ]

                )



                if context_values.shape[0] != context_length:

                    continue



                if future_values.shape[0] != prediction_length:

                    continue



                self.samples.append(

                    {

                    "context":
                        context_values,


                    "future":
                        future_values,


                    "glacier":
                        glacier,


                    "target_datetime":
                        dates[i]

                    }

                )



        if len(self.samples) == 0:

            raise ValueError(
                f"No windows created "
                f"for {target_split}, context {context_length}"
            )



    def __len__(self):

        return len(self.samples)



    def __getitem__(self, index):

        return self.samples[index]

# Create datasets for all contexts
raw_datasets = {}


print()

print("="*70)

print("CREATING FORECAST WINDOWS")

print("="*70)



for ctx in CONTEXTS:


    print()

    print(
        "CONTEXT:",
        ctx
    )



    train_dataset = GlacierWindowDataset(
        all_df,
        ctx,
        PREDICTION_LENGTH,
        "train"
    )


    val_dataset = GlacierWindowDataset(
        all_df,
        ctx,
        PREDICTION_LENGTH,
        "validation"
    )


    test_dataset = GlacierWindowDataset(
        all_df,
        ctx,
        PREDICTION_LENGTH,
        "test"
    )



    raw_datasets[ctx] = {

        "train": train_dataset,

        "validation": val_dataset,

        "test": test_dataset

    }



    print(
        "Train windows:",
        len(train_dataset)
    )


    print(
        "Validation windows:",
        len(val_dataset)
    )


    print(
        "Test windows:",
        len(test_dataset)
    )


# Load pretrained Moirai converter

print()



print("LOADING MOIRAI MODULE")





converter_module = (
    MoiraiModule
    .from_pretrained(
        MODEL_NAME
    )
)



print("Moirai module loaded")

# Create Moirai converters
moirai_converters = {}



for ctx in CONTEXTS:


    moirai_converters[ctx] = MoiraiForecast(

        module=converter_module,

        prediction_length=PREDICTION_LENGTH,

        target_dim=NUM_VARIATES,

        feat_dynamic_real_dim=0,

        past_feat_dynamic_real_dim=0,

        context_length=ctx,

        patch_size=PATCH_SIZE,

        num_samples=NUM_SAMPLES

    )



    print(
        "Converter created:",
        ctx
    )


# Moirai collate function
def collate_moirai(
    batch,
    context_length
):


    converter = (
        moirai_converters[
            context_length
        ]
    )



    past_target = torch.tensor(

        np.stack(
            [
                x["context"]
                for x in batch
            ]
        ),

        dtype=torch.float32

    )



    future_target = torch.tensor(

        np.stack(
            [
                x["future"]
                for x in batch
            ]
        ),

        dtype=torch.float32

    )



    batch_size = past_target.shape[0]



    past_observed_target = torch.ones(
        past_target.shape,
        dtype=torch.bool
    )


    future_observed_target = torch.ones(
        future_target.shape,
        dtype=torch.bool
    )


    past_is_pad = torch.zeros(
        batch_size,
        context_length,
        dtype=torch.bool
    )


    future_is_pad = torch.zeros(
        batch_size,
        PREDICTION_LENGTH,
        dtype=torch.bool
    )



    (

        target,

        observed_mask,

        sample_id,

        time_id,

        variate_id,

        prediction_mask

    ) = converter._convert(

        PATCH_SIZE,

        past_target=past_target,

        past_observed_target=past_observed_target,

        past_is_pad=past_is_pad,

        future_target=future_target,

        future_observed_target=future_observed_target,

        future_is_pad=future_is_pad

    )



    return {

        "target": target,

        "observed_mask": observed_mask,

        "sample_id": sample_id,

        "time_id": time_id,

        "variate_id": variate_id,

        "prediction_mask": prediction_mask,

        "patch_size":
            torch.full_like(
                sample_id,
                PATCH_SIZE,
                dtype=torch.long
            ),

        "future_target":
            future_target

    }

# DataLoader helper
def create_moirai_loader(
    dataset,
    context_length,
    shuffle=False
):


    return DataLoader(

        dataset,

        batch_size=BATCH_SIZE,

        shuffle=shuffle,

        num_workers=NUM_WORKERS,

        pin_memory=False,

        collate_fn=lambda batch:
            collate_moirai(
                batch,
                context_length
            )

    )



print()



print("PART 2 COMPLETED SUCCESSFULLY")

print("MOIRAI FINE-TUNING TRAINING")

# Create fine-tuning model


def create_finetune_model(context_length):


    print()

    print(
        "Loading pretrained Moirai for context:",
        context_length
    )



    module = (
        MoiraiModule
        .from_pretrained(
            MODEL_NAME
        )
    )


    print(
        "Pretrained module loaded"
    )



    loss_function = PackedNLLLoss()



    estimated_steps = (
        EPOCHS *
        MAX_TRAIN_BATCHES_PER_EPOCH
    )



    model = MoiraiFinetune(

        module=module,

        min_patches=2,

        min_mask_ratio=0.0,

        max_mask_ratio=0.0,

        max_dim=MAX_PATCH_SIZE,


        num_training_steps=estimated_steps,

        num_warmup_steps=0,


        loss_func=loss_function,


        num_samples=NUM_SAMPLES,


        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY,


        context_length=context_length,

        prediction_length=PREDICTION_LENGTH,

        patch_size=PATCH_SIZE,


        finetune_pattern="full"

    )


    return model.to(DEVICE)

# Train one context

def train_one_context(context_length):


    print()

    print("="*70)

    print(
        f"TRAINING CONTEXT {context_length}"
    )

    print("="*70)



    train_dataset = (
        raw_datasets[context_length]["train"]
    )


    val_dataset = (
        raw_datasets[context_length]["validation"]
    )



    train_loader = create_moirai_loader(

        train_dataset,

        context_length,

        shuffle=True

    )


    val_loader = create_moirai_loader(

        val_dataset,

        context_length,

        shuffle=False

    )



    print()

    print(
        "Training samples:",
        len(train_dataset)
    )

    print(
        "Validation samples:",
        len(val_dataset)
    )



    model = create_finetune_model(
        context_length
    )



    total_parameters = sum(
        p.numel()
        for p in model.parameters()
    )


    trainable_parameters = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )



    print()

    print(
        "Total parameters:",
        f"{total_parameters:,}"
    )


    print(
        "Trainable parameters:",
        f"{trainable_parameters:,}"
    )



    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY

    )



    history = {

        "epoch": [],

        "train_loss": [],

        "val_loss": []

    }
 # Epoch loop
   
    for epoch in range(
        1,
        EPOCHS + 1
    ):


        model.train()


        train_losses = []



        for batch_idx, batch in enumerate(train_loader):



            batch = {

                k:
                (
                    v.to(DEVICE)
                    if torch.is_tensor(v)
                    else v
                )

                for k,v in batch.items()

            }



            optimizer.zero_grad(
                set_to_none=True
            )



            loss = model.training_step(

                batch,

                batch_idx

            )



            if not torch.isfinite(loss):

                raise RuntimeError(
                    f"Invalid loss: {loss}"
                )



            loss.backward()



            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                max_norm=1.0

            )



            optimizer.step()



            train_losses.append(

                float(
                    loss.detach()
                    .cpu()
                )

            )



            if (
                MAX_TRAIN_BATCHES_PER_EPOCH
                is not None
                and
                batch_idx + 1 >= MAX_TRAIN_BATCHES_PER_EPOCH
            ):

                break



        train_loss = np.mean(
            train_losses
        )
        # Validation
        
        model.eval()


        val_losses = []



        with torch.no_grad():


            for val_idx, batch in enumerate(val_loader):


                batch = {

                    k:
                    (
                        v.to(DEVICE)
                        if torch.is_tensor(v)
                        else v
                    )

                    for k,v in batch.items()

                }



                loss = model.validation_step(

                    batch,

                    val_idx

                )



                if torch.isfinite(loss):

                    val_losses.append(

                        float(
                            loss.cpu()
                        )

                    )



                if (

                    MAX_VAL_BATCHES is not None

                    and

                    val_idx + 1 >= MAX_VAL_BATCHES

                ):

                    break




        val_loss = np.mean(
            val_losses
        )



        history["epoch"].append(
            epoch
        )


        history["train_loss"].append(
            train_loss
        )


        history["val_loss"].append(
            val_loss
        )



        print()

        print(

            f"Epoch {epoch}/{EPOCHS} | "

            f"Train Loss: {train_loss:.6f} | "

            f"Val Loss: {val_loss:.6f}"

        )



    return model, history

# Train all contexts

training_models = {}

training_histories = {}



print()

print("="*70)

print("STARTING ALL CONTEXT TRAINING")

print("="*70)



for ctx in CONTEXTS:


    model, history = train_one_context(
        ctx
    )


    training_models[ctx] = model


    training_histories[ctx] = history



    print()

    print(
        "COMPLETED CONTEXT:",
        ctx
    )



print()

print("ALL MOIRAI CONTEXTS TRAINED")


print("MOIRAI EVALUATION")

def get_moirai_predictions(
    model,
    loader,
    context_length,
    max_batches=None
):


    model.eval()


    all_predictions = []

    all_targets = []



    context_token_length = math.ceil(
        context_length / PATCH_SIZE
    )


    prediction_token_length = math.ceil(
        PREDICTION_LENGTH / PATCH_SIZE
    )



    target_token_index = (

        context_token_length *
        NUM_VARIATES

        +

        TARGET_VARIATE_INDEX *
        prediction_token_length

    )



    print()

    print(
        "Context:",
        context_length
    )

    print(
        "Target token index:",
        target_token_index
    )



    with torch.no_grad():


        for batch_idx, batch in enumerate(loader):



            batch = {

                k:
                (
                    v.to(DEVICE)
                    if torch.is_tensor(v)
                    else v
                )

                for k,v in batch.items()

            }



            distribution = model(

                target=batch["target"],

                observed_mask=batch["observed_mask"],

                sample_id=batch["sample_id"],

                time_id=batch["time_id"],

                variate_id=batch["variate_id"],

                prediction_mask=batch["prediction_mask"],

                patch_size=batch["patch_size"]

            )



            # Monte Carlo samples

            samples = distribution.sample(

                torch.Size(
                    [NUM_SAMPLES]
                )

            )



            # Select target variable

            prediction_samples = (

                samples[

                    :,

                    :,

                    target_token_index,

                    0

                ]

            )



            # Median forecast

            predictions = torch.median(

                prediction_samples,

                dim=0

            ).values



            targets = (

                batch["future_target"]

                [

                    :,

                    0,

                    TARGET_VARIATE_INDEX

                ]

            )



            all_predictions.append(

                predictions.cpu().numpy()

            )


            all_targets.append(

                targets.cpu().numpy()

            )



            if (

                max_batches is not None

                and

                batch_idx + 1 >= max_batches

            ):

                break





    predictions = np.concatenate(
        all_predictions
    )


    targets = np.concatenate(
        all_targets
    )



    return targets, predictions

# Metrics
def calculate_metrics(
    y_true,
    y_pred
):


    mse = mean_squared_error(
        y_true,
        y_pred
    )


    rmse = np.sqrt(
        mse
    )


    mae = mean_absolute_error(
        y_true,
        y_pred
    )


    r2 = r2_score(
        y_true,
        y_pred
    )



    return {

        "MAE": mae,

        "MSE": mse,

        "RMSE": rmse,

        "R2": r2,

        "N": len(y_true)

    }

# Validation evaluation
validation_results = {}

validation_predictions = {}

validation_targets = {}



print()

print("="*70)

print("VALIDATION RESULTS")

print("="*70)



for ctx in CONTEXTS:


    print()

    print(
        "Validation Context:",
        ctx
    )



    val_loader = create_moirai_loader(

        raw_datasets[ctx]["validation"],

        ctx,

        shuffle=False

    )



    y_true, y_pred = get_moirai_predictions(

        training_models[ctx],

        val_loader,

        ctx,

        max_batches=MAX_VAL_BATCHES

    )



    metrics = calculate_metrics(

        y_true,

        y_pred

    )



    validation_results[ctx] = metrics

    validation_targets[ctx] = y_true

    validation_predictions[ctx] = y_pred



    print()

    print(
        metrics
    )


# Validation table

validation_table = pd.DataFrame(

    [

        {

            "Context":ctx,

            **validation_results[ctx]

        }

        for ctx in CONTEXTS

    ]

)



print()

print("="*70)

print("VALIDATION TABLE")

print("="*70)



display(validation_table)



best_context = int(

    validation_table

    .sort_values(
        "RMSE"
    )

    .iloc[0]["Context"]

)



print()

print(
    "BEST CONTEXT:",
    best_context
)


# Final Test Evaluation
test_results = {}

test_predictions = {}

test_targets = {}



print()

print("="*70)

print("FINAL TEST RESULTS")

print("="*70)



for ctx in CONTEXTS:


    print()

    print(
        "Testing Context:",
        ctx
    )



    test_loader = create_moirai_loader(

        raw_datasets[ctx]["test"],

        ctx,

        shuffle=False

    )



    y_true, y_pred = get_moirai_predictions(

        training_models[ctx],

        test_loader,

        ctx,

        max_batches=MAX_TEST_BATCHES

    )



    metrics = calculate_metrics(

        y_true,

        y_pred

    )



    test_results[ctx] = metrics

    test_targets[ctx] = y_true

    test_predictions[ctx] = y_pred



    print()

    print(metrics)


# Final Test Table
test_table = pd.DataFrame(

    [

        {

            "Context":ctx,

            **test_results[ctx]

        }

        for ctx in CONTEXTS

    ]

)



print()

print("="*70)

print("FINAL TEST PERFORMANCE")

print("="*70)



display(test_table)



print()

print(
    "BEST VALIDATION CONTEXT:",
    best_context
)


print(
    "BEST TEST RESULT:"
)

display(

    test_table[
        test_table["Context"] == best_context
    ]

)

In [ ]:
# Chronos fine tuned
import warnings
warnings.filterwarnings("ignore")

import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from chronos import Chronos2Pipeline


SEED = 42

ID_COL = "glacier"
TIME_COL = "datetime"
TARGET_COL = "retreat_change_next_month"

DATA_DIR = Path("/home/parcot1/imputed_data")

TRAIN_FILE = DATA_DIR / "D3_train_encoded.csv"
VAL_FILE = DATA_DIR / "D3_validation_encoded.csv"
TEST_FILE = DATA_DIR / "D3_test_encoded.csv"

FINETUNE_DIR = Path(
    "/home/parcot1/data/chronos2_glacier_finetuned"
)

CONTEXT_LENGTHS = [6, 12, 24, 36]
PREDICTION_LENGTH = 1
BATCH_SIZE = 32

CONFIGS = [
    (200, 1e-5),
    (500, 1e-5),
]

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

gc.collect()

FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available() and torch.cuda.device_count() == 0:
    print("No usable CUDA device found. Using CPU.")
    torch.cuda.is_available = lambda: False


def preprocess(df):
    df = df.copy()

    df[TIME_COL] = pd.to_datetime(
        df[TIME_COL],
        errors="coerce"
    )

    df[TARGET_COL] = pd.to_numeric(
        df[TARGET_COL],
        errors="coerce"
    )

    df[ID_COL] = (
        df[ID_COL]
        .astype(str)
        .str.strip()
    )

    df = df.dropna(
        subset=[
            ID_COL,
            TIME_COL,
            TARGET_COL
        ]
    )

    df = df.drop_duplicates(
        subset=[
            ID_COL,
            TIME_COL
        ],
        keep="last"
    )

    return df.sort_values(
        [ID_COL, TIME_COL]
    ).reset_index(drop=True)


def add_datetime_features(df, global_min_date):
    df = df.copy()

    df["year"] = df[TIME_COL].dt.year.astype(np.float32)

    df["month_num"] = df[TIME_COL].dt.month.astype(
        np.float32
    )

    df["month_sin"] = np.sin(
        2 * np.pi * df["month_num"] / 12.0
    ).astype(np.float32)

    df["month_cos"] = np.cos(
        2 * np.pi * df["month_num"] / 12.0
    ).astype(np.float32)

    df["time_idx"] = (
        (df[TIME_COL].dt.year - global_min_date.year) * 12
        + (df[TIME_COL].dt.month - global_min_date.month)
    ).astype(np.float32)

    return df


def create_training_series(df, predictor_columns):
    series = []

    for _, glacier_df in df.groupby(ID_COL):
        glacier_df = glacier_df.sort_values(
            TIME_COL
        ).reset_index(drop=True)

        if len(glacier_df) <= PREDICTION_LENGTH:
            continue

        past_covariates = {
            column: glacier_df[column].to_numpy(
                dtype=np.float32
            )
            for column in predictor_columns
        }

        series.append(
            {
                "target": glacier_df[TARGET_COL].to_numpy(
                    dtype=np.float32
                ),
                "past_covariates": past_covariates
            }
        )

    return series


def create_chronos_input(
    history_df,
    glacier_id,
    prediction_date,
    context_length,
    predictor_columns
):
    glacier_history = history_df[
        history_df[ID_COL] == glacier_id
    ].copy()

    glacier_history = glacier_history[
        glacier_history[TIME_COL] < prediction_date
    ].copy()

    glacier_history = glacier_history.sort_values(
        TIME_COL
    )

    if len(glacier_history) < context_length:
        return None

    glacier_history = glacier_history.tail(
        context_length
    ).copy()

    past_covariates = {
        column: glacier_history[column].to_numpy(
            dtype=np.float32
        )
        for column in predictor_columns
    }

    return {
        "target": glacier_history[TARGET_COL].to_numpy(
            dtype=np.float32
        ),
        "past_covariates": past_covariates
    }


def predict_one_step(
    pipeline,
    history_df,
    glacier_id,
    prediction_date,
    context_length,
    predictor_columns
):
    model_input = create_chronos_input(
        history_df=history_df,
        glacier_id=glacier_id,
        prediction_date=prediction_date,
        context_length=context_length,
        predictor_columns=predictor_columns
    )

    if model_input is None:
        return None

    with torch.no_grad():
        forecast = pipeline.predict(
            inputs=[model_input],
            prediction_length=PREDICTION_LENGTH
        )

    return forecast


def extract_prediction(forecast):
    if isinstance(forecast, list):
        if len(forecast) == 0:
            raise ValueError("Forecast list is empty.")

        forecast = forecast[0]

    if isinstance(forecast, np.ndarray):
        forecast = torch.from_numpy(forecast)

    if not isinstance(forecast, torch.Tensor):
        forecast = torch.tensor(forecast)

    forecast = forecast.detach().cpu()

    if forecast.numel() == 0:
        raise ValueError("Forecast tensor is empty.")

    if forecast.ndim == 3:
        return float(
            torch.median(
                forecast[0, :, 0]
            ).item()
        )

    if forecast.ndim == 2:
        return float(
            torch.median(
                forecast[:, 0]
            ).item()
        )

    if forecast.ndim == 1:
        return float(
            torch.median(forecast).item()
        )

    if forecast.ndim == 0:
        return float(forecast.item())

    raise ValueError(
        f"Unexpected forecast shape: {tuple(forecast.shape)}"
    )


def run_rolling_forecast(
    pipeline,
    initial_history,
    forecast_data,
    context_length,
    period_name,
    predictor_columns
):
    history = initial_history.copy()

    forecast_data = forecast_data.sort_values(
        [ID_COL, TIME_COL]
    ).copy()

    results = []

    prediction_dates = sorted(
        forecast_data[TIME_COL]
        .dropna()
        .unique()
    )

    for date_index, prediction_date in enumerate(
        prediction_dates,
        start=1
    ):
        prediction_date = pd.Timestamp(prediction_date)

        print(
            f"{period_name}: "
            f"{date_index}/{len(prediction_dates)} "
            f"{prediction_date.date()}",
            end="\r"
        )

        current_rows = forecast_data[
            forecast_data[TIME_COL] == prediction_date
        ].copy()

        for glacier_id in current_rows[ID_COL].unique():
            current_row = current_rows[
                current_rows[ID_COL] == glacier_id
            ]

            actual_value = float(
                current_row[TARGET_COL].iloc[0]
            )

            glacier_history = history[
                history[ID_COL] == glacier_id
            ]

            glacier_history = glacier_history[
                glacier_history[TIME_COL] < prediction_date
            ]

            if len(glacier_history) < context_length:
                continue

            try:
                forecast = predict_one_step(
                    pipeline=pipeline,
                    history_df=history,
                    glacier_id=glacier_id,
                    prediction_date=prediction_date,
                    context_length=context_length,
                    predictor_columns=predictor_columns
                )

                if forecast is None:
                    continue

                predicted_value = extract_prediction(
                    forecast
                )

                results.append(
                    {
                        ID_COL: glacier_id,
                        TIME_COL: prediction_date,
                        "actual": actual_value,
                        "prediction": predicted_value,
                        "context_length": context_length,
                        "period": period_name
                    }
                )

            except Exception as error:
                print(
                    f"\nPrediction error | "
                    f"glacier={glacier_id} | "
                    f"date={prediction_date.date()} | "
                    f"error={error}"
                )

        history = pd.concat(
            [history, current_rows],
            ignore_index=True
        )

        history = history.sort_values(
            [ID_COL, TIME_COL]
        ).reset_index(drop=True)

    print()

    return pd.DataFrame(results)


def calculate_metrics(predictions_df):
    if predictions_df.empty:
        return {
            "MAE": np.nan,
            "MSE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "N": 0
        }

    y_true = predictions_df["actual"].to_numpy()
    y_pred = predictions_df["prediction"].to_numpy()

    mse = mean_squared_error(
        y_true,
        y_pred
    )

    return {
        "MAE": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "MSE": float(mse),
        "RMSE": float(np.sqrt(mse)),
        "R2": float(r2_score(y_true, y_pred)),
        "N": int(len(y_true))
    }


train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)
test_df = pd.read_csv(TEST_FILE)

print("Original train shape:", train_df.shape)
print("Original validation shape:", val_df.shape)
print("Original test shape:", test_df.shape)

train_df = preprocess(train_df)
val_df = preprocess(val_df)
test_df = preprocess(test_df)

print("Train range:", train_df[TIME_COL].min(), "to", train_df[TIME_COL].max())
print("Validation range:", val_df[TIME_COL].min(), "to", val_df[TIME_COL].max())
print("Test range:", test_df[TIME_COL].min(), "to", test_df[TIME_COL].max())

if train_df.empty or val_df.empty or test_df.empty:
    raise ValueError(
        "A dataset split is empty after preprocessing."
    )

all_glaciers = sorted(
    pd.concat(
        [
            train_df[[ID_COL]],
            val_df[[ID_COL]],
            test_df[[ID_COL]]
        ],
        ignore_index=True
    )[ID_COL].unique()
)

glacier_codes = {
    glacier: index
    for index, glacier in enumerate(all_glaciers)
}

for df in [train_df, val_df, test_df]:
    df["glacier_code"] = (
        df[ID_COL]
        .map(glacier_codes)
        .astype(np.float32)
    )

global_min_date = train_df[TIME_COL].min()

train_df = add_datetime_features(
    train_df,
    global_min_date
)

val_df = add_datetime_features(
    val_df,
    global_min_date
)

test_df = add_datetime_features(
    test_df,
    global_min_date
)

exclude_columns = {
    ID_COL,
    TIME_COL,
    TARGET_COL
}

all_df = pd.concat(
    [
        train_df,
        val_df,
        test_df
    ],
    ignore_index=True
)

predictor_columns = [
    column
    for column in all_df.columns
    if column not in exclude_columns
    and pd.api.types.is_numeric_dtype(all_df[column])
]

print("\nPredictor columns:")
for column in predictor_columns:
    print("-", column)

for column in predictor_columns:
    train_df[column] = pd.to_numeric(
        train_df[column],
        errors="coerce"
    )

    val_df[column] = pd.to_numeric(
        val_df[column],
        errors="coerce"
    )

    test_df[column] = pd.to_numeric(
        test_df[column],
        errors="coerce"
    )

model_columns = [
    ID_COL,
    TIME_COL,
    TARGET_COL
] + predictor_columns

train_df = train_df.dropna(
    subset=model_columns
).reset_index(drop=True)

val_df = val_df.dropna(
    subset=model_columns
).reset_index(drop=True)

test_df = test_df.dropna(
    subset=model_columns
).reset_index(drop=True)

print("\nCleaned train shape:", train_df.shape)
print("Cleaned validation shape:", val_df.shape)
print("Cleaned test shape:", test_df.shape)

train_series = create_training_series(
    train_df,
    predictor_columns
)

if len(train_series) == 0:
    raise ValueError(
        "No valid training series were created."
    )

print("\nNumber of training series:", len(train_series))
print("First target length:", len(train_series[0]["target"]))
print(
    "Number of past covariates:",
    len(train_series[0]["past_covariates"])
)

all_validation_results = []

for steps, learning_rate in CONFIGS:
    print("\nFitting model")
    print("num_steps:", steps)
    print("learning_rate:", learning_rate)

    pipeline = Chronos2Pipeline.from_pretrained(
        "amazon/chronos-2",
        device_map="cpu"
    )

    learning_rate_name = str(
        learning_rate
    ).replace(".", "p")

    output_dir = (
        FINETUNE_DIR
        / f"steps_{steps}_lr_{learning_rate_name}"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    pipeline = pipeline.fit(
        inputs=train_series,
        prediction_length=PREDICTION_LENGTH,
        num_steps=steps,
        learning_rate=learning_rate,
        batch_size=BATCH_SIZE,
        output_dir=output_dir,
        disable_data_parallel=True
    )

    config_results = []

    for context_length in CONTEXT_LENGTHS:
        validation_predictions_df = run_rolling_forecast(
            pipeline=pipeline,
            initial_history=train_df,
            forecast_data=val_df,
            context_length=context_length,
            period_name=(
                f"Validation | steps={steps} | "
                f"lr={learning_rate} | "
                f"context={context_length}"
            ),
            predictor_columns=predictor_columns
        )

        metrics = calculate_metrics(
            validation_predictions_df
        )

        result = {
            "num_steps": steps,
            "learning_rate": learning_rate,
            "ContextLength": context_length,
            "MAE": metrics["MAE"],
            "MSE": metrics["MSE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
            "N": metrics["N"],
            "model_dir": str(output_dir)
        }

        config_results.append(result)
        all_validation_results.append(result)

        validation_predictions_df.to_csv(
            output_dir
            / f"validation_predictions_context_{context_length}.csv",
            index=False
        )

    config_results_df = pd.DataFrame(
        config_results
    ).sort_values(
        "ContextLength"
    ).reset_index(drop=True)

    print("\nValidation metrics")
    print(config_results_df.to_string(index=False))

    del pipeline
    gc.collect()

all_validation_results_df = pd.DataFrame(
    all_validation_results
).sort_values(
    ["RMSE", "MAE"]
).reset_index(drop=True)

all_validation_results_df.to_csv(
    FINETUNE_DIR / "all_validation_results.csv",
    index=False
)

print("\nAll validation results")
print(all_validation_results_df.to_string(index=False))

valid_results_df = all_validation_results_df.dropna(
    subset=["RMSE"]
).reset_index(drop=True)

if valid_results_df.empty:
    raise ValueError(
        "No valid validation metrics were produced."
    )

best_config = valid_results_df.iloc[0]

best_steps = int(best_config["num_steps"])
best_learning_rate = float(
    best_config["learning_rate"]
)
best_context_length = int(
    best_config["ContextLength"]
)

print("\nBest validation configuration")
print("num_steps:", best_steps)
print("learning_rate:", best_learning_rate)
print("context_length:", best_context_length)
print("validation RMSE:", best_config["RMSE"])

train_val_df = pd.concat(
    [
        train_df,
        val_df
    ],
    ignore_index=True
).sort_values(
    [ID_COL, TIME_COL]
).reset_index(drop=True)

train_val_series = create_training_series(
    train_val_df,
    predictor_columns
)

final_model_dir = (
    FINETUNE_DIR
    / (
        f"final_steps_{best_steps}_"
        f"lr_{str(best_learning_rate).replace('.', 'p')}"
    )
)

final_model_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("\nRetraining best model")

best_pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"
)

best_pipeline = best_pipeline.fit(
    inputs=train_val_series,
    prediction_length=PREDICTION_LENGTH,
    num_steps=best_steps,
    learning_rate=best_learning_rate,
    batch_size=BATCH_SIZE,
    output_dir=final_model_dir,
    disable_data_parallel=True
)

print("\nTest evaluation")

test_predictions_df = run_rolling_forecast(
    pipeline=best_pipeline,
    initial_history=train_val_df,
    forecast_data=test_df,
    context_length=best_context_length,
    period_name=(
        f"Test | steps={best_steps} | "
        f"lr={best_learning_rate} | "
        f"context={best_context_length}"
    ),
    predictor_columns=predictor_columns
)

test_metrics = calculate_metrics(
    test_predictions_df
)

test_results_df = pd.DataFrame(
    [
        {
            "num_steps": best_steps,
            "learning_rate": best_learning_rate,
            "ContextLength": best_context_length,
            "MAE": test_metrics["MAE"],
            "MSE": test_metrics["MSE"],
            "RMSE": test_metrics["RMSE"],
            "R2": test_metrics["R2"],
            "N": test_metrics["N"]
        }
    ]
)

test_predictions_df.to_csv(
    FINETUNE_DIR / "final_test_predictions.csv",
    index=False
)

test_results_df.to_csv(
    FINETUNE_DIR / "final_test_results.csv",
    index=False
)

print("\nFinal test results")
print(test_results_df.to_string(index=False))

In [ ]:
# persistence

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = "/home/parcot1/updated_data"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
VALIDATION_PATH = os.path.join(DATA_DIR, "validation.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")


target_col = "retreat_change_next_month"
id_col = "glacier"
time_col = "datetime"
context_lengths = [6, 12, 24, 36]


train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)


required_columns = [id_col, time_col, target_col]
for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")


def basic_prep(df):
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df[id_col] = df[id_col].astype(str).str.strip()
    df = df.dropna(subset=[id_col, time_col, target_col]).copy()
    df = df.sort_values([id_col, time_col]).reset_index(drop=True)
    return df


train_df = basic_prep(train_df)
val_df = basic_prep(val_df)
test_df = basic_prep(test_df)

train_ids = set(train_df[id_col].unique())
val_ids = set(val_df[id_col].unique())
test_ids = set(test_df[id_col].unique())
common_ids = train_ids & val_ids & test_ids

print("\nTrain glaciers:", len(train_ids))
print("Validation glaciers:", len(val_ids))
print("Test glaciers:", len(test_ids))
print("Common glaciers:", len(common_ids))

train_panel = train_df[train_df[id_col].isin(common_ids)].copy()
val_panel = val_df[val_df[id_col].isin(common_ids)].copy()
test_panel = test_df[test_df[id_col].isin(common_ids)].copy()

def rolling_persistence_forecast(history_df, future_df, context_length, id_col, time_col, target_col):
    preds = []

    history_groups = {
        gid: g.sort_values(time_col).reset_index(drop=True).copy()
        for gid, g in history_df.groupby(id_col, sort=True)
    }
    future_groups = {
        gid: g.sort_values(time_col).reset_index(drop=True).copy()
        for gid, g in future_df.groupby(id_col, sort=True)
    }

    common_gids = sorted(set(history_groups.keys()) & set(future_groups.keys()))

    for gid in common_gids:
        hist = history_groups[gid]
        fut = future_groups[gid]

        hist_targets = hist[target_col].to_numpy(dtype=np.float32)

        if len(hist_targets) < context_length:
            continue

        for i in range(len(fut)):
            if len(hist_targets) < 1:
                continue

            pred = float(hist_targets[-1])
            actual = float(fut.iloc[i][target_col])

            if np.isnan(pred) or np.isnan(actual):
                continue

            preds.append({
                id_col: str(gid),
                time_col: fut.iloc[i][time_col],
                "actual": actual,
                "predicted": pred
            })

            new_target = np.array([actual], dtype=np.float32)
            hist_targets = np.concatenate([hist_targets, new_target])

    return pd.DataFrame(preds)


def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if len(y_true) == 0:
        return {
            "RMSE": np.nan,
            "MSE": np.nan,
            "MAE": np.nan,
            "R2": np.nan,
            "N": 0
        }

    mse = mean_squared_error(y_true, y_pred)

    return {
        "RMSE": float(np.sqrt(mse)),
        "MSE": float(mse),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 and np.var(y_true) > 0 else np.nan,
        "N": int(len(y_true))
    }

validation_results_all = []
test_results_all = []
window_counts = []

for context_length in context_lengths:
    print(f"\nRunning Persistence with context length = {context_length}")

    val_results = rolling_persistence_forecast(
        history_df=train_panel,
        future_df=val_panel,
        context_length=context_length,
        id_col=id_col,
        time_col=time_col,
        target_col=target_col
    )

    if not val_results.empty:
        val_metrics = calculate_metrics(val_results["actual"], val_results["predicted"])
        validation_results_all.append({
            "Model": "Persistence",
            "Split": "Validation",
            "ContextLength": context_length,
            **val_metrics
        })
        print("Validation metrics:", val_metrics)

    train_val_history = (
        pd.concat([train_panel, val_panel], ignore_index=True)
        .sort_values([id_col, time_col])
        .reset_index(drop=True)
    )

    test_results = rolling_persistence_forecast(
        history_df=train_val_history,
        future_df=test_panel,
        context_length=context_length,
        id_col=id_col,
        time_col=time_col,
        target_col=target_col
    )

    if not test_results.empty:
        test_metrics = calculate_metrics(test_results["actual"], test_results["predicted"])
        test_results_all.append({
            "Model": "Persistence",
            "Split": "Test_Comparison_Only",
            "ContextLength": context_length,
            **test_metrics
        })
        print("Test comparison metrics:", test_metrics)

    window_counts.append({
        "ContextLength": context_length,
        "ValidationWindows": len(val_results),
        "ValidationGlaciers": val_results[id_col].nunique() if not val_results.empty else 0,
        "TestWindows": len(test_results),
        "TestGlaciers": test_results[id_col].nunique() if not test_results.empty else 0
    })


validation_metrics_df = pd.DataFrame(validation_results_all)
test_metrics_df = pd.DataFrame(test_results_all)
window_counts_df = pd.DataFrame(window_counts)

if not validation_metrics_df.empty:
    validation_metrics_df = validation_metrics_df.sort_values("RMSE").reset_index(drop=True)
    best_context = int(validation_metrics_df.iloc[0]["ContextLength"])
else:
    best_context = None

print("\nValidation comparison")
if not validation_metrics_df.empty:
    print(validation_metrics_df[["ContextLength", "RMSE", "MSE", "MAE", "R2", "N"]])

print("\nBest context based on validation RMSE:", best_context)

print("\nTest comparison results")
if not test_metrics_df.empty:
    test_metrics_df = test_metrics_df.sort_values("ContextLength").reset_index(drop=True)
    print(test_metrics_df[["ContextLength", "RMSE", "MSE", "MAE", "R2", "N"]])

print("\nWindow counts")
print(window_counts_df)


final_test_df = None

if best_context is not None:
    train_val_df = (
        pd.concat([train_panel, val_panel], ignore_index=True)
        .sort_values([id_col, time_col])
        .reset_index(drop=True)
    )

    final_test_results = rolling_persistence_forecast(
        history_df=train_val_df,
        future_df=test_panel,
        context_length=best_context,
        id_col=id_col,
        time_col=time_col,
        target_col=target_col
    )

    if not final_test_results.empty:
        final_test_metrics = calculate_metrics(
            final_test_results["actual"],
            final_test_results["predicted"]
        )

        final_test_df = pd.DataFrame([{
            "Model": "Persistence",
            "Split": "Official_Final_Test",
            "ContextLength": best_context,
            **final_test_metrics
        }])

        print("\nOfficial final test results")
        print(final_test_df[["ContextLength", "RMSE", "MSE", "MAE", "R2", "N"]])


print("\nPersistence experiment complete")
print("Target:", target_col)
print("Series ID:", id_col)
print("Context lengths:", context_lengths)
print("Best context:", best_context)

In [ ]:
# xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Paths

DATA_DIR = "/home/parcot1/updated_data"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
VALIDATION_PATH = os.path.join(DATA_DIR, "validation.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

# Settings

target_col = "retreat_change_next_month"
id_col = "glacier"
time_col = "datetime"
context_lengths = [6, 12, 24, 36]
SEED = 42


# Load data

print("Load data")

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)


# Prepare data

def basic_prep(df):
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col, target_col]).copy()
    df[id_col] = df[id_col].astype(str).str.strip()
    df = df.sort_values([id_col, time_col]).reset_index(drop=True)
    return df


train_df = basic_prep(train_df)
val_df = basic_prep(val_df)
test_df = basic_prep(test_df)


# Glacier code
all_glaciers = sorted(
    pd.concat([train_df[[id_col]], val_df[[id_col]], test_df[[id_col]]], ignore_index=True)[id_col]
    .dropna()
    .unique()
)

glacier_codes = {glacier: idx for idx, glacier in enumerate(all_glaciers)}

print("Number of unique glaciers:", len(glacier_codes))


# Time features

global_min_date = min(
    train_df[time_col].min(),
    val_df[time_col].min(),
    test_df[time_col].min()
)

def add_engineered_features(df):
    df = df.copy()
    df["glacier_code"] = df[id_col].map(glacier_codes).astype(int)
    df["year"] = df[time_col].dt.year.astype(int)
    df["month_num"] = df[time_col].dt.month.astype(int)
    df["month_sin"] = np.sin(2 * np.pi * df["month_num"] / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * df["month_num"] / 12.0)
    df["time_idx"] = (
        (df[time_col].dt.year - global_min_date.year) * 12
        + (df[time_col].dt.month - global_min_date.month)
    ).astype(int)
    return df


train_df = add_engineered_features(train_df)
val_df = add_engineered_features(val_df)
test_df = add_engineered_features(test_df)


# Diagnostics

print("Dataset diagnostics")
print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("Train glaciers:", train_df[id_col].nunique())
print("Validation glaciers:", val_df[id_col].nunique())
print("Test glaciers:", test_df[id_col].nunique())

print("Train range:", train_df[time_col].min(), "to", train_df[time_col].max())
print("Validation range:", val_df[time_col].min(), "to", val_df[time_col].max())
print("Test range:", test_df[time_col].min(), "to", test_df[time_col].max())


# Predictors

excluded_columns = {target_col, id_col, time_col}

feature_cols = [
    col for col in train_df.columns
    if col not in excluded_columns and pd.api.types.is_numeric_dtype(train_df[col])
]

print("Feature information")
print("Target variable:", target_col)
print("Series ID:", id_col)
print("Time variable:", time_col)
print("Number of numeric predictors:", len(feature_cols))

print("Feature columns used:")
for i, col in enumerate(feature_cols, start=1):
    print(f"{i}. {col}")


# Missing values

train_medians = train_df[feature_cols].median()

def fill_missing_values(df):
    df = df.copy()
    df[feature_cols] = df[feature_cols].fillna(train_medians)
    return df


train_df = fill_missing_values(train_df)
val_df = fill_missing_values(val_df)
test_df = fill_missing_values(test_df)


# Training windows

def create_training_windows(data, context):
    X = []
    y = []
    metadata = []

    for glacier_id, g in data.groupby(id_col, sort=False):
        g = g.sort_values(time_col).reset_index(drop=True)

        features = g[feature_cols].values.astype(np.float32)
        targets = g[target_col].values.astype(np.float32)
        dates = g[time_col].values

        for i in range(context, len(g)):
            X_window = features[i - context:i].reshape(-1)
            X.append(X_window)
            y.append(targets[i])
            metadata.append({
                id_col: glacier_id,
                time_col: pd.Timestamp(dates[i])
            })

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        pd.DataFrame(metadata)
    )


# Rolling forecast windows

def create_rolling_forecast_windows(history_df, forecast_df, context):
    X = []
    y = []
    metadata = []

    for glacier_id in forecast_df[id_col].unique():
        history_g = (
            history_df[history_df[id_col] == glacier_id]
            .sort_values(time_col)
            .reset_index(drop=True)
        )

        forecast_g = (
            forecast_df[forecast_df[id_col] == glacier_id]
            .sort_values(time_col)
            .reset_index(drop=True)
        )

        history_features = history_g[feature_cols].values.astype(np.float32)
        forecast_features = forecast_g[feature_cols].values.astype(np.float32)
        forecast_targets = forecast_g[target_col].values.astype(np.float32)

        for j in range(len(forecast_g)):
            available_history = np.vstack([history_features, forecast_features[:j]])

            if len(available_history) < context:
                continue

            X_window = available_history[-context:].reshape(-1)
            X.append(X_window)
            y.append(forecast_targets[j])
            metadata.append({
                id_col: glacier_id,
                time_col: forecast_g.loc[j, time_col]
            })

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        pd.DataFrame(metadata)
    )


# Metrics

def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
        "N": len(y_true)
    }


# XGBoost model

def create_xgb_model():
    return XGBRegressor(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=SEED,
        n_jobs=-1
    )


# Validation

print("Validation")

validation_results = []
validation_predictions = []

for context in context_lengths:
    print(f"Validation context {context}")

    X_train, y_train, _ = create_training_windows(train_df, context)
    X_val, y_val, meta_val = create_rolling_forecast_windows(
        history_df=train_df,
        forecast_df=val_df,
        context=context
    )

    print("Training windows:", len(X_train))
    print("Validation windows:", len(X_val))

    if len(X_train) == 0 or len(X_val) == 0:
        print("Skipping context:", context)
        continue

    model = create_xgb_model()
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    val_metrics = calculate_metrics(y_val, val_pred)

    validation_results.append({
        "Model": "XGBoost",
        "ContextLength": context,
        "ValidationMAE": val_metrics["MAE"],
        "ValidationMSE": val_metrics["MSE"],
        "ValidationRMSE": val_metrics["RMSE"],
        "ValidationR2": val_metrics["R2"],
        "ValidationN": val_metrics["N"]
    })

    val_output = meta_val.copy()
    val_output["actual"] = y_val
    val_output["predicted"] = val_pred
    val_output["ContextLength"] = context
    validation_predictions.append(val_output)


validation_results_df = pd.DataFrame(validation_results).sort_values("ValidationRMSE").reset_index(drop=True)

print("Validation results")
print(validation_results_df.to_string(index=False))


# Best context

best_context = int(validation_results_df.iloc[0]["ContextLength"])
best_validation_rmse = validation_results_df.iloc[0]["ValidationRMSE"]

print("Best context:", best_context)
print("Best validation RMSE:", best_validation_rmse)


# Combine train and validation

train_val_df = pd.concat([train_df, val_df], ignore_index=True)
train_val_df = train_val_df.sort_values([id_col, time_col]).reset_index(drop=True)


# Test

print("Test")

test_results = []
test_predictions = []

for context in context_lengths:
    print(f"Test context {context}")

    X_train, y_train, _ = create_training_windows(train_df, context)
    X_val, y_val, _ = create_rolling_forecast_windows(
        history_df=train_df,
        forecast_df=val_df,
        context=context
    )

    if len(X_train) == 0 or len(X_val) == 0:
        print("Skipping context:", context)
        continue

    if len(X_val) > 0:
        X_train_val = np.vstack([X_train, X_val])
        y_train_val = np.concatenate([y_train, y_val])
    else:
        X_train_val = X_train
        y_train_val = y_train

    model = create_xgb_model()
    model.fit(X_train_val, y_train_val)

    X_test, y_test, meta_test = create_rolling_forecast_windows(
        history_df=train_val_df,
        forecast_df=test_df,
        context=context
    )

    print("Test windows:", len(X_test))

    if len(X_test) == 0:
        print("Skipping context:", context)
        continue

    test_pred = model.predict(X_test)
    test_metrics = calculate_metrics(y_test, test_pred)

    validation_row = validation_results_df[validation_results_df["ContextLength"] == context]
    validation_rmse = validation_row["ValidationRMSE"].iloc[0] if not validation_row.empty else np.nan
    selected_by_validation = context == best_context

    test_results.append({
        "Model": "XGBoost",
        "ContextLength": context,
        "ValidationRMSE": validation_rmse,
        "TestMAE": test_metrics["MAE"],
        "TestMSE": test_metrics["MSE"],
        "TestRMSE": test_metrics["RMSE"],
        "TestR2": test_metrics["R2"],
        "TestN": test_metrics["N"],
        "SelectedByValidation": selected_by_validation
    })

    test_output = meta_test.copy()
    test_output["actual"] = y_test
    test_output["predicted"] = test_pred
    test_output["ContextLength"] = context
    test_output["SelectedByValidation"] = selected_by_validation
    test_predictions.append(test_output)


test_results_df = pd.DataFrame(test_results).sort_values("ContextLength").reset_index(drop=True)

print("Test results")
print(test_results_df.to_string(index=False))


# Official result

official_result = test_results_df[test_results_df["ContextLength"] == best_context].iloc[0]

print("Selected context:", best_context)
print("Validation RMSE:", best_validation_rmse)
print("Official Test MAE:", official_result["TestMAE"])
print("Official Test MSE:", official_result["TestMSE"])
print("Official Test RMSE:", official_result["TestRMSE"])
print("Official Test R2:", official_result["TestR2"])
print("Official Test N:", official_result["TestN"])


# Final summary

print("Final summary")
print(
    test_results_df[
        ["ContextLength", "ValidationRMSE", "TestRMSE", "TestMAE", "TestR2", "SelectedByValidation"]
    ].to_string(index=False)
)

print("\nBest context selected using Validation RMSE:", best_context)
print("Best Validation RMSE:", best_validation_rmse)
print("Official Test RMSE:", official_result["TestRMSE"])

In [ ]:
# TFT
# TFT_GLUONTS

# Imports and setup
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from gluonts.dataset.common import ListDataset
from gluonts.torch.model.tft import TemporalFusionTransformerEstimator

# Paths and config

data_dir = "/home/parcot1/updated_data"
train_path = os.path.join(data_dir, "train.csv")
val_path = os.path.join(data_dir, "validation.csv")
test_path = os.path.join(data_dir, "test.csv")

target_col = "retreat_change_next_month"
id_col = "glacier"
time_col = "datetime"
series_id_col = "glacier_code"

freq = "M"
CONTEXT_LENGTHS = [6, 12, 24, 36]
PREDICTION_LENGTH = 1

BATCH_SIZE = 16
NUM_BATCHES_PER_EPOCH = 50
EPOCHS = 50
LEARNING_RATE = 1e-5
SEED = 42

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Metric calculation function
def calc_metrics(model_name, split_name, context_length, y_true, y_pred, evaluation_type):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mse = mean_squared_error(y_true, y_pred)
    return {
        "model": model_name,
        "split": split_name,
        "evaluation_type": evaluation_type,
        "context_length": context_length,
        "RMSE": float(np.sqrt(mse)),
        "MSE": float(mse),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 and np.var(y_true) > 0 else np.nan,
        "n_obs": int(len(y_true))
    }

# Data preparation functions
def minimal_prep(df):
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df = df.dropna(subset=[time_col, target_col]).copy()
    df[id_col] = df[id_col].astype(str).str.strip()
    df = df.sort_values([id_col, time_col]).reset_index(drop=True)
    return df

def add_datetime_features(df, global_min_date):
    df = df.copy()
    df["year"] = df[time_col].dt.year.astype(np.int32)
    df["month_num"] = df[time_col].dt.month.astype(np.int32)
    df["month_sin"] = np.sin(2 * np.pi * df["month_num"] / 12.0).astype(np.float32)
    df["month_cos"] = np.cos(2 * np.pi * df["month_num"] / 12.0).astype(np.float32)
    df["time_idx"] = (
        (df[time_col].dt.year - global_min_date.year) * 12
        + (df[time_col].dt.month - global_min_date.month)
    ).astype(np.int32)
    df["time_idx_scaled"] = (df["time_idx"] / 12.0).astype(np.float32)
    return df

# Feature matrix builders
def build_past_dynamic_matrix(g, cols):
    if len(cols) == 0:
        return None
    mats = []
    for c in cols:
        vals = pd.to_numeric(g[c], errors="coerce").to_numpy(dtype=np.float32)
        vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
        mats.append(vals)
    return np.vstack(mats).astype(np.float32)

def build_known_dynamic_matrix(g):
    return np.vstack([
        g["month_sin"].to_numpy(dtype=np.float32),
        g["month_cos"].to_numpy(dtype=np.float32),
        g["month_num"].to_numpy(dtype=np.float32),
        g["year"].to_numpy(dtype=np.float32),
        g["time_idx_scaled"].to_numpy(dtype=np.float32)
    ]).astype(np.float32)

# Dataset construction functions
def make_train_dataset(data, context_length, past_covariate_cols):
    entries = []
    for gid, g in data.groupby(series_id_col):
        g = g.sort_values(time_col).reset_index(drop=True)
        if len(g) < (context_length + PREDICTION_LENGTH):
            continue
        if g[target_col].isna().any():
            continue

        target = g[target_col].to_numpy(dtype=np.float32)
        past_features = build_past_dynamic_matrix(g, past_covariate_cols)
        known_features = build_known_dynamic_matrix(g)

        if past_features is None:
            continue
        if past_features.shape[1] != len(target):
            continue
        if known_features.shape[1] != len(target):
            continue

        entry = {
            "start": pd.Period(g[time_col].iloc[0], freq=freq),
            "target": target,
            "item_id": str(gid),
            "feat_static_cat": np.array([int(gid)], dtype=np.int64),
            "feat_dynamic_real": known_features,
            "past_feat_dynamic_real": past_features
        }
        entries.append(entry)

    return ListDataset(entries, freq=freq)

def make_one_step_entry(history_g, future_row, context_length, past_covariate_cols):
    hist = history_g.sort_values(time_col).tail(context_length).copy()
    if len(hist) != context_length:
        return None

    target = hist[target_col].to_numpy(dtype=np.float32)
    past_features = build_past_dynamic_matrix(hist, past_covariate_cols)
    if past_features is None:
        return None

    hist_known = build_known_dynamic_matrix(hist)
    future_known = np.array([
        [np.float32(future_row["month_sin"])],
        [np.float32(future_row["month_cos"])],
        [np.float32(future_row["month_num"])],
        [np.float32(future_row["year"])],
        [np.float32(future_row["time_idx_scaled"])]
    ], dtype=np.float32)

    known_features = np.concatenate([hist_known, future_known], axis=1)

    return {
        "start": pd.Period(hist[time_col].iloc[0], freq=freq),
        "target": target,
        "item_id": str(hist[series_id_col].iloc[0]),
        "feat_static_cat": np.array([int(hist[series_id_col].iloc[0])], dtype=np.int64),
        "feat_dynamic_real": known_features,
        "past_feat_dynamic_real": past_features
    }

# Rolling forecast function
def rolling_forecast(history_df, future_df, context_length, past_covariate_cols, predictor, split_name):
    results = []
    glaciers = sorted(future_df[series_id_col].dropna().unique())

    for gid in glaciers:
        history_g = history_df[history_df[series_id_col] == gid].sort_values(time_col).copy()
        future_g = future_df[future_df[series_id_col] == gid].sort_values(time_col).copy()

        if len(history_g) < context_length or len(future_g) == 0:
            continue

        for i in range(len(future_g)):
            current_row = future_g.iloc[i]
            context_df = history_g.sort_values(time_col).tail(context_length).copy()
            if len(context_df) < context_length:
                continue

            entry = make_one_step_entry(
                history_g=context_df,
                future_row=current_row,
                context_length=context_length,
                past_covariate_cols=past_covariate_cols
            )
            if entry is None:
                continue

            try:
                rolling_ds = ListDataset([entry], freq=freq)
                forecast = next(predictor.predict(rolling_ds))
                pred = float(forecast.quantile(0.5)[0])

                results.append({
                    "glacier_code": int(gid),
                    "glacier": current_row[id_col] if id_col in current_row.index else None,
                    "datetime": current_row[time_col],
                    "actual": float(current_row[target_col]),
                    "predicted": pred,
                    "split": split_name,
                    "context_length": context_length
                })

                history_g = pd.concat([history_g, current_row.to_frame().T], ignore_index=True)
                history_g = history_g.sort_values(time_col).reset_index(drop=True)

            except Exception as e:
                print(f"Skipped glacier_code={gid} at {current_row[time_col]}: {e}")

    return pd.DataFrame(results)

# Load and prepare data
print("Loading data")

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

train_df = minimal_prep(train_df)
val_df = minimal_prep(val_df)
test_df = minimal_prep(test_df)

all_glaciers = sorted(
    pd.concat([train_df[[id_col]], val_df[[id_col]], test_df[[id_col]]], ignore_index=True)[id_col]
    .dropna()
    .astype(str)
    .unique()
)
glacier_code_map = {g: i for i, g in enumerate(all_glaciers)}

train_df[series_id_col] = train_df[id_col].map(glacier_code_map).astype(int)
val_df[series_id_col] = val_df[id_col].map(glacier_code_map).astype(int)
test_df[series_id_col] = test_df[id_col].map(glacier_code_map).astype(int)

global_min_date = min(
    train_df[time_col].min(),
    val_df[time_col].min(),
    test_df[time_col].min()
)

train_df = add_datetime_features(train_df, global_min_date)
val_df = add_datetime_features(val_df, global_min_date)
test_df = add_datetime_features(test_df, global_min_date)

print("Train:", len(train_df), "rows |", train_df[series_id_col].nunique(), "glaciers")
print("Validation:", len(val_df), "rows |", val_df[series_id_col].nunique(), "glaciers")
print("Test:", len(test_df), "rows |", test_df[series_id_col].nunique(), "glaciers")

# Variable selection
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

exclude_cols = {
    target_col, id_col, series_id_col, time_col,
    "year", "month_num", "month_sin", "month_cos",
    "time_idx", "time_idx_scaled"
}

past_covariate_cols = [
    c for c in all_df.columns
    if c not in exclude_cols and pd.api.types.is_numeric_dtype(all_df[c])
]

print("Target:", target_col)
print("Series ID:", series_id_col)
print("Past covariates:", len(past_covariate_cols))
for c in past_covariate_cols:
    print(" -", c)

print("Known time features: month_sin, month_cos, month_num, year, time_idx_scaled")

# Stage 1: context selection
all_metrics = []
all_predictions = []
window_counts = []

print("Stage 1: context selection")

for ctx in CONTEXT_LENGTHS:
    print("Context:", ctx)

    train_ds = make_train_dataset(train_df, ctx, past_covariate_cols)
    train_entries = list(train_ds)

    if len(train_entries) == 0:
        print("Skipped: no valid training series")
        continue

    print("Training series:", len(train_entries))

    val_history = train_df.copy()
    test_history = (
        pd.concat([train_df, val_df], ignore_index=True)
        .sort_values([series_id_col, time_col])
        .reset_index(drop=True)
    )

    accelerator = "gpu" if torch.cuda.is_available() else "cpu"

    estimator = TemporalFusionTransformerEstimator(
        freq=freq,
        prediction_length=PREDICTION_LENGTH,
        context_length=ctx,
        hidden_dim=32,
        variable_dim=16,
        num_heads=4,
        dropout_rate=0.1,
        batch_size=BATCH_SIZE,
        num_batches_per_epoch=NUM_BATCHES_PER_EPOCH,
        lr=LEARNING_RATE,
        quantiles=[0.1, 0.5, 0.9],
        static_cardinalities=[len(glacier_code_map)],
        dynamic_dims=[1, 1, 1, 1, 1],
        past_dynamic_dims=[1] * len(past_covariate_cols) if len(past_covariate_cols) > 0 else [],
        trainer_kwargs={
            "max_epochs": EPOCHS,
            "accelerator": accelerator,
            "devices": 1,
            "enable_progress_bar": False,
            "logger": False
        }
    )

    print("Training")
    predictor = estimator.train(training_data=train_ds)

    print("Validation")
    val_results = rolling_forecast(
        history_df=val_history,
        future_df=val_df,
        context_length=ctx,
        past_covariate_cols=past_covariate_cols,
        predictor=predictor,
        split_name="validation"
    )

    print("Test")
    test_results = rolling_forecast(
        history_df=test_history,
        future_df=test_df,
        context_length=ctx,
        past_covariate_cols=past_covariate_cols,
        predictor=predictor,
        split_name="test_all_contexts"
    )

    window_counts.append({
        "context_length": ctx,
        "train_series": len(train_entries),
        "validation_windows": len(val_results),
        "validation_glaciers": val_results["glacier_code"].nunique() if not val_results.empty else 0,
        "test_windows": len(test_results),
        "test_glaciers": test_results["glacier_code"].nunique() if not test_results.empty else 0
    })

    if not val_results.empty:
        all_metrics.append(
            calc_metrics("TFT_GluonTS", "validation", ctx, val_results["actual"], val_results["predicted"], "context_selection")
        )
        all_predictions.append(val_results)

    if not test_results.empty:
        all_metrics.append(
            calc_metrics("TFT_GluonTS", "test_all_contexts", ctx, test_results["actual"], test_results["predicted"], "comparison_only")
        )
        all_predictions.append(test_results)

# Select best context
metrics_df = pd.DataFrame(all_metrics)
predictions_df = pd.concat(all_predictions, ignore_index=True) if len(all_predictions) > 0 else pd.DataFrame()
window_counts_df = pd.DataFrame(window_counts)

validation_metrics = (
    metrics_df[metrics_df["split"] == "validation"]
    .sort_values("RMSE")
    .reset_index(drop=True)
)

if not validation_metrics.empty:
    best_ctx = int(validation_metrics.iloc[0]["context_length"])
    best_val_rmse = float(validation_metrics.iloc[0]["RMSE"])
else:
    best_ctx = None
    best_val_rmse = None

print("Validation results")
print(validation_metrics)
print("Best context:", best_ctx)
print("Best validation RMSE:", best_val_rmse)

# Stage 2: final model training
final_test_results = pd.DataFrame()

if best_ctx is not None:
    print("Stage 2: final model")
    print("Best context:", best_ctx)

    train_val_df = (
        pd.concat([train_df, val_df], ignore_index=True)
        .sort_values([series_id_col, time_col])
        .reset_index(drop=True)
    )

    final_train_ds = make_train_dataset(train_val_df, best_ctx, past_covariate_cols)
    final_train_entries = list(final_train_ds)
    print("Final training series:", len(final_train_entries))

    accelerator = "gpu" if torch.cuda.is_available() else "cpu"

    final_estimator = TemporalFusionTransformerEstimator(
        freq=freq,
        prediction_length=PREDICTION_LENGTH,
        context_length=best_ctx,
        hidden_dim=32,
        variable_dim=16,
        num_heads=4,
        dropout_rate=0.1,
        batch_size=BATCH_SIZE,
        num_batches_per_epoch=NUM_BATCHES_PER_EPOCH,
        lr=LEARNING_RATE,
        quantiles=[0.1, 0.5, 0.9],
        static_cardinalities=[len(glacier_code_map)],
        dynamic_dims=[1, 1, 1, 1, 1],
        past_dynamic_dims=[1] * len(past_covariate_cols) if len(past_covariate_cols) > 0 else [],
        trainer_kwargs={
            "max_epochs": EPOCHS,
            "accelerator": accelerator,
            "devices": 1,
            "enable_progress_bar": False,
            "logger": False
        }
    )

    print("Training final model")
    final_predictor = final_estimator.train(training_data=final_train_ds)

    print("Running final test")
    final_test_results = rolling_forecast(
        history_df=train_val_df,
        future_df=test_df,
        context_length=best_ctx,
        past_covariate_cols=past_covariate_cols,
        predictor=final_predictor,
        split_name="test_final_best_context"
    )

# Final metrics
if not final_test_results.empty:
    final_test_metric = calc_metrics(
        model_name="TFT_GluonTS",
        split_name="test_final_best_context",
        context_length=best_ctx,
        y_true=final_test_results["actual"],
        y_pred=final_test_results["predicted"],
        evaluation_type="official_final_test"
    )
    metrics_df = pd.concat([metrics_df, pd.DataFrame([final_test_metric])], ignore_index=True)
    predictions_df = pd.concat([predictions_df, final_test_results], ignore_index=True)

metrics_df = metrics_df.sort_values(["split", "context_length"]).reset_index(drop=True)

# Print results
print("All results")
print(metrics_df.to_string(index=False))

print("Validation comparison")
val_compare = metrics_df[metrics_df["split"] == "validation"]
if not val_compare.empty:
    print(val_compare[["context_length", "RMSE", "MAE", "R2", "n_obs"]].sort_values("RMSE").to_string(index=False))

print("Test comparison")
test_compare = metrics_df[metrics_df["split"] == "test_all_contexts"]
if not test_compare.empty:
    print(test_compare[["context_length", "RMSE", "MAE", "R2", "n_obs"]].sort_values("context_length").to_string(index=False))
else:
    print("No test comparison results")

print("Final test")
official_test = metrics_df[metrics_df["split"] == "test_final_best_context"]
if not official_test.empty:
    print(official_test[["context_length", "RMSE", "MAE", "MSE", "R2", "n_obs"]].to_string(index=False))
else:
    print("Final test results were not generated")

print("Window counts")
print(window_counts_df.to_string(index=False))

# Summary
print("Done")
print("Target:", target_col)
print("Series ID:", series_id_col)
print("Number of past covariates:", len(past_covariate_cols))
print("Context lengths tested:", CONTEXT_LENGTHS)
print("Prediction length:", PREDICTION_LENGTH)
print("Best context:", best_ctx)
print("Best validation RMSE:", best_val_rmse)